# A/B testing tools: a reference by capability

> **Development version.** This notebook uses the A/B testing tools that ship with
> retentioneering 5.4.0 (`get_ab_test`, `get_ab_bootstrap`, `get_ab_paths`,
> `get_sample_ratio`, `get_ab_transitions`, `get_ab_steps`, the `ab_test`, `ab_transitions` and
> `ab_steps` widgets, funnel intervals, the A/B MCP tools and
> `datasets.load_ab_test`). Until that release is on PyPI it runs against the development
> install:
>
> ```bash
> git clone https://github.com/retentioneering/retentioneering-tools
> cd retentioneering-tools
> make install && make build   # needs uv and Node.js: dependencies, then the widget bundle
> pip install -e .             # into the environment of your notebook kernel
> ```

The [tutorial](ab_testing_tutorial.ipynb) reads one experiment as a story. This notebook is
the companion reference: it is organised by **capability**, so you can jump to "how do I do
X" and see the call, its output and how to read it. Every public feature of the A/B tools is
exercised at least once, each on the bundled dataset where it means something: a broken split
for the sample ratio check, a session-level assignment for the unit guard, an A/A test for
calibration, a trial for horizons, a push campaign for intention to treat.

| Section | What it shows |
|---|---|
| [1. Datasets](#1.-The-bundled-datasets) | `list_ab_tests`, `load_ab_test` (every name, `as_dataframe`, `with_truth`, `n_users`, `seed`), `load_ab_test_assignment` |
| [2. Sample ratio](#2.-Sample-ratio-mismatch:-get_sample_ratio) | `get_sample_ratio`: plain, `assignment`, `expected_shares`, `strata_col`, `period`, `reading`, `mismatch`, `silent_paths`, `multi_valued_paths` |
| [3. Scorecard](#3.-The-scorecard:-get_ab_test) | every metric kind, `test`, `correction`, `correction_scope`, guardrails, `ci_level`, `power`, `min_support`, every `support_warning`, `strata_col` with `strata_pre_exposure`, `population`, `conditional`, `reading`, `diff` forms, id matching, `path_col` and the unit guard |
| [4. Time](#4.-Time:-periods,-horizon,-novelty) | `period` (calendar and since assignment), `period_population`, `horizon`, `get_ab_bootstrap(contrast=)`, `bootstrap_weights`, `n_bootstrap="auto"`, `random_state` |
| [5. Per-path table](#5.-The-per-path-table:-get_ab_paths) | `get_ab_paths`: what the scorecard aggregates, and what to do with it |
| [6. Funnels](#6.-Funnels-with-intervals) | `funnel_data(diff=, ci_level=, correction=)`, overall vs conditional steps, `get_conversion_rate(diff=)` |
| [7. Edges and steps](#7.-Which-edges-and-steps-differ:-get_ab_transitions-and-get_ab_steps) | `get_ab_transitions` (every `edge_weight`, `edges=`, `horizon`, `correction`), `get_ab_steps`, `conditional`, `transition_graph(ab_transitions=)`, the `ab_transitions` / `ab_steps` matrices and `get_ab_transitions_table` / `get_ab_steps_table` |
| [8. Widget](#8.-The-ab_test-widget) | `ab_test`, `ab_test_data`, `state_file`, `height`, `sidebar_open` |
| [9. MCP](#9.-Asking-an-agent:-the-MCP-tools) | `serve(assignment=)` and the A/B tools an agent calls |
| [10. Blind clusters](#10.-Blind-pre-exposure-clusters-as-strata) | clusters on the pre-exposure window as declared strata |
| [11. Checks](#11.-Checks) | asserts every claim the prose makes |

**Conventions.** `delta` is always group 1 minus group 2, here treatment minus control;
intervals are 95 % unless a cell says otherwise; 0/1 metrics are shares, so their deltas are
read in percentage points (pp). The datasets are synthetic and deterministic: every number
quoted in the text is printed by the cell above it, and the last cell asserts them.
Bootstrap calls use `n_bootstrap=200` (the constant `N_BOOT`) to keep the notebook fast; the
library default is 1,000, which is what you should use for a real readout. Widgets are shown
with `render_static()` so that they survive an HTML export; drop that call to get the live
widget, whose sidebar recomputes.

In [ ]:
import json
import os
import tempfile
import warnings
from pathlib import Path

os.environ["RETENTIONEERING_NO_TRACK"] = "1"

import numpy as np
import pandas as pd
from scipy import stats

import retentioneering as rete
from retentioneering import datasets

if not hasattr(rete.Eventstream, "get_ab_transitions"):
    raise ImportError(
        "This notebook needs retentioneering >= 5.4.0 or the development install "
        "(see the first cell)."
    )
print("Retentioneering", rete.__version__)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4f}".format)

DIFF = ("exp_group", "treatment", "control")  # delta = treatment - control
N_BOOT = 200  # bootstrap replicates in this notebook; the library default is 1000

# Metric configs reused below: registry configs with a row name.
ORDERS = {"metric": "event_count", "metric_args": {"event": "order_success"}}
CONVERSION = {
    "name": "conversion",
    "metric": "has_event",
    "metric_args": {"event": "order_success"},
}
REVENUE = {
    "name": "revenue_per_user",
    "metric": "column_sum",
    "metric_args": {"column": "order_value"},
}
AOV = {
    "name": "aov",
    "metric": "column_sum",
    "metric_args": {"column": "order_value"},
    "denominator": ORDERS,
}
CORE = [
    "metric",
    "group1_value",
    "group2_value",
    "delta",
    "delta_ci_low",
    "delta_ci_high",
    "p_value_adjusted",
    "mde",
    "verdict",
]


def call(func, *args, **kwargs):
    # Run a library call and print its warnings as plain lines, so they read as output.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = func(*args, **kwargs)
    for warning in caught:
        print(f"{warning.category.__name__}: {warning.message}")
    return result


def show_error(func, *args, **kwargs):
    # Run a call that must fail and print the error the library raises.
    try:
        call(func, *args, **kwargs)
    except Exception as error:  # the deliberate failures of this notebook
        print(f"{type(error).__name__}: {error}")
        return
    raise AssertionError("the call was expected to raise")


def pp(value):
    # A share difference in percentage points.
    return f"{value * 100:+.2f} pp"

## 1. The bundled datasets

`datasets.list_ab_tests()` names the six synthetic experiments with a one-line description.
The descriptions say what each test is, not what it hides: each generator's docstring
(`retentioneering.datasets.ab_test` for the default, `retentioneering.datasets.ab_cases.<name>`
for the others) describes the design and ends with a "Solutions (spoilers)" section. The first
entry is the default of `load_ab_test()`.

In [ ]:
catalogue = datasets.list_ab_tests()
pd.Series(catalogue, name="description").rename_axis("name").to_frame()

### 1.1 The default dataset and its assignment table

`load_ab_test()` returns an `Eventstream` of "Northwind Home": about 10,000 users of a
home-goods store, a 50/50 test of a one-tap `quick_add` button. `load_ab_test_assignment()`
returns the experiment's **assignment table**: one row per assigned user (`user_id`), the `arm`
and `assigned_at`. The table is the population of the experiment; the stream holds only users
who produced at least one event.

In [ ]:
stream = datasets.load_ab_test()  # name="northwind_home"
assignment = datasets.load_ab_test_assignment()

print("path columns:", stream.schema.path_cols)
print(
    "segments:",
    stream.schema.segment_cols,
    "| custom columns:",
    stream.schema.custom_cols,
)
print(
    "users in the stream:",
    stream.df["user_id"].nunique(),
    "| rows in the table:",
    len(assignment),
)
assignment.head()

`as_dataframe=True` returns the raw events frame instead of an `Eventstream`, and
`with_truth=True` adds the planted truth column (here `user_type`: `scanner`, `curator` or
`other`). The truth is for checking an analysis at the end, never for the analysis itself;
section 10 uses it that way. `n_users=` and `seed=` generate another draw of the same design:
the generation is deterministic for a given `(name, n_users, seed)`, and the table of
`load_ab_test_assignment` must be asked for with the same values.

In [ ]:
events = datasets.load_ab_test(as_dataframe=True, with_truth=True)
print(type(events).__name__, events.shape)
planted_types = events.groupby("user_id")["user_type"].first()
print(
    "planted user types:", planted_types.value_counts(normalize=True).round(3).to_dict()
)

# n_users / seed: the same arguments give the same data, another seed another draw
draw = datasets.load_ab_test(n_users=500, seed=7, as_dataframe=True)
same = datasets.load_ab_test(n_users=500, seed=7, as_dataframe=True)
other = datasets.load_ab_test(n_users=500, seed=8, as_dataframe=True)
draw_assignment = datasets.load_ab_test_assignment(n_users=500, seed=7)
print(
    "same seed, same data:",
    draw.equals(same),
    "| another seed, same data:",
    draw.equals(other),
)
print(
    "users in the 500-user draw:",
    draw["user_id"].nunique(),
    "of",
    len(draw_assignment),
    "assigned",
)
events.head(3)

### 1.2 The five other experiments

Each name loads the same way; each has its own assignment table, unit and truth column. The
table below loads all of them once (about a second each) and keeps them for the sections that
use them. Note the assignment unit of `session_assignment`: its table is indexed by
`session_id`, because that experiment assigned sessions, not users.

In [ ]:
CASES = {}
rows = []
for name in list(catalogue)[1:]:
    case_stream = datasets.load_ab_test(name)
    case_assignment = datasets.load_ab_test_assignment(name)
    with_truth = datasets.load_ab_test(name, as_dataframe=True, with_truth=True)
    CASES[name] = (case_stream, case_assignment)
    rows.append(
        {
            "dataset": name,
            "events": len(case_stream.df),
            "paths with events": case_stream.df["user_id"].nunique(),
            "table rows": len(case_assignment),
            "table index": case_assignment.index.name,
            "table columns": ", ".join(case_assignment.columns),
            "segments": ", ".join(case_stream.schema.segment_cols),
            "truth column": ", ".join(
                sorted(set(with_truth.columns) - set(case_stream.df.columns))
            ),
        }
    )
bug, bug_assignment = CASES["srm_platform_bug"]
sessions, session_table = CASES["session_assignment"]
aa, aa_assignment = CASES["aa_calibration"]
trial, trial_assignment = CASES["trial_conversion"]
push, push_assignment = CASES["push_campaign"]
pd.DataFrame(rows).set_index("dataset")

Every section below says which dataset it uses and which planted story it finds:

| Dataset | Planted story | Where |
|---|---|---|
| `northwind_home` | inconclusive top line, opposite effects in pre-exposure user types, an effect that fades with exposure age | 3, 4.1-4.4, 5, 6, 7, 9 |
| `srm_platform_bug` | a logging bug loses Android treatment users: SRM, and a fake lift | 2.2 |
| `session_assignment` | sessions were assigned, users analysed: two arms per user, intervals too narrow | 2.3, 3.12 |
| `aa_calibration` | an A/A test: a clean null, except six whale accounts of one reseller | 4.6 |
| `trial_conversion` | a paywall test read too early: immature cohorts fake a lift | 4.5 |
| `push_campaign` | assignment outside the product: only an ITT read over the table is the experiment | 2.4, 3.10, 4.7 |

## 2. Sample ratio mismatch: `get_sample_ratio`

A sample ratio mismatch (SRM), arms whose sizes differ from the design by more than chance,
means the assignment, the logging or a filter lost users from one arm. It is checked **before
any metric**: if it fails, no metric of the test can be trusted. `get_sample_ratio` runs a
chi-square goodness-of-fit test of the counts per arm against the expected shares (50/50 by
default) and returns a dict:

- `levels`: one row per arm with `paths`, `share`, `expected_share`, `expected_paths`;
- `chi2`, `dof`, `p_value`, `alpha` (0.001 by default: stricter than 0.05, because an SRM
  alarm gates every other result);
- `mismatch`: `p_value < alpha` **or** any path carrying more than one arm;
- `multi_valued_paths`: paths with two arm values (on the stream, or disagreeing with the
  table), an assignment bug whatever the shares say;
- `silent_paths`: assigned paths with no event in the stream (with `assignment=` only);
- `unassigned_paths`: paths not counted (not in the table, no arm, an arm outside
  `expected_shares`);
- `basis`: what was counted; `reading`: the check in one sentence.

### 2.1 On the default dataset: plain, with the table, with expected shares, by group

In [ ]:
srm_stream = stream.get_sample_ratio("exp_group")  # the paths of the stream
srm = stream.get_sample_ratio(
    "exp_group", assignment=assignment
)  # the assignment table
for label, result in [("stream", srm_stream), ("assignment table", srm)]:
    print(f"{label:>16}: {result['basis']}")
    print(f"{'':>16}  {result['reading']}")
    print(
        f"{'':>16}  mismatch={result['mismatch']}, silent_paths={result['silent_paths']}, "
        f"multi_valued_paths={result['multi_valued_paths']}, unassigned_paths={result['unassigned_paths']}"
    )
srm["levels"]

Both counts match 50/50. The table counts 10,000 users, 267 of them silent (assigned, never
seen in the log); the stream counts only the 9,733 users with events. Run the check on the
table when you have one: a logging bug that loses users from one arm is invisible on the
stream if the lost users never reached it.

`expected_shares=` declares another design. The same data read against a 60/40 design is a
mismatch: the `reading` says so and that the test is invalid. `strata_col=` repeats the check
within a variable fixed at assignment (signup platform), and `period=` within calendar units of
`assigned_at` (`"D"`, `"W"`, `"M"`, `"Q"`, `"Y"`), each with Holm-adjusted p-values over the
groups. The overall `mismatch` does not include the groups: read the `strata` / `periods`
tables or the `reading`.

In [ ]:
wrong_design = stream.get_sample_ratio(
    "exp_group", expected_shares={"control": 0.6, "treatment": 0.4}
)
print("60/40:", wrong_design["reading"])

by_platform = stream.get_sample_ratio(
    "exp_group", assignment=assignment, strata_col="platform"
)
by_week = stream.get_sample_ratio(
    assignment=assignment, period="W"
)  # segment_col is optional with a table
print("by platform:", by_platform["reading"])
print("by week:", by_week["reading"])
display(by_platform["strata"])
by_week["periods"]

The `<MISSING>` stratum holds the silent users: Northwind's table has no platform column, so a
user without events has no platform. `levels` with `strata_col` or `period` gets a leading
`stratum` / `period` column, with the overall rows marked `<ALL>`.

### 2.2 `srm_platform_bug`: a mismatch, located

A checkout redesign on iOS, Android and web. The analyst's table is the analytics exposure log,
which comes from the same pipeline as the events.

In [ ]:
bug_srm = bug.get_sample_ratio(assignment=bug_assignment, strata_col="platform")
print(bug_srm["reading"])
display(bug_srm["strata"])

bug_weeks = bug.get_sample_ratio(assignment=bug_assignment, period="W")
print(bug_weeks["reading"])
bug_daily = bug.get_sample_ratio(assignment=bug_assignment, period="D")
bug_days = bug_daily["levels"][bug_daily["levels"]["period"] != "<ALL>"]
bug_days = bug_days.pivot(index="period", columns="level", values="paths")
treatment_share = bug_days["treatment"] / bug_days.sum(axis=1)
print(
    f"by day: {len(bug_daily['periods'])} days, {int(bug_daily['periods']['mismatch'].sum())} flagged after Holm; "
    f"treatment share before 2026-05-12: {treatment_share[treatment_share.index < '2026-05-12'].mean():.2f}, "
    f"from 2026-05-12 on: {treatment_share[treatment_share.index >= '2026-05-12'].mean():.2f}"
)
bug_weeks["periods"]

The table is far off 50/50 (5,040 control against 3,990 treatment), and the strata put the
whole mismatch on Android; iOS and web are balanced. By week, the first week is balanced and
every later one is not; by day, the treatment share of the table drops from 0.50 to 0.42 from
2026-05-12 on, yet with 28 daily tests no single day is flagged at the strict Holm-adjusted
alpha: a weekly grain is the sharper tool here. The generator's solution: the Android treatment
build released on day 9 logs a user only after some time in the app, so users who leave
quickly are never logged, neither their events nor their exposure.

What the broken test would have said if read anyway, and what the planted truth says:

In [ ]:
naive = bug.get_ab_test(DIFF, [CONVERSION], assignment=bug_assignment)
by_bug_platform = call(
    bug.get_ab_test,
    DIFF,
    [CONVERSION],
    assignment=bug_assignment,
    strata_col="platform",
    strata_pre_exposure=True,
)
display(naive[CORE])
display(by_bug_platform[["stratum"] + CORE])

bug_truth = datasets.load_ab_test(
    "srm_platform_bug", as_dataframe=True, with_truth=True
)
bug_users = bug_truth.groupby("user_id").agg(
    intent=("intent", "first"),
    arm=("exp_group", "first"),
    converted=("event", lambda e: bool((e == "order_success").any())),
)
bug_users.pivot_table(index="intent", columns="arm", values="converted", aggfunc="mean")

The naive scorecard reads `increase` (about +3 pp) and the lift sits on Android only. The
planted truth shows why it is fake: within each intent level, logged users buy at the same rate
in both arms; the lift comes from the low-intent Android treatment users the log lost. The
right answer is not to read this test at all: fix the logging and rerun.

### 2.3 `session_assignment`: `multi_valued_paths`

The experiment service assigned an arm at the start of every **session**; the product question
is per user. At the user grain, a user whose sessions saw both cards carries two arms.

In [ ]:
per_user = sessions.get_sample_ratio("exp_group")  # user_id is the stream's path column
print("users:", per_user["reading"])
per_session = sessions.get_sample_ratio(assignment=session_table, path_col="session_id")
print("sessions:", per_session["reading"])

The user-grain check raises the alarm through `multi_valued_paths` (2,454 users carry both arms),
although the shares are fine (p 0.83). At the session grain, with the service's table, the split
is a clean 50/50. Section 3.12 shows what the scorecard does with this experiment.

### 2.4 `push_campaign`: when a stream "mismatch" is the treatment working

A marketing push with a 50/50 holdout. The table lists 20,000 users; the stream holds only
those who opened the app.

In [ ]:
push_stream_srm = push.get_sample_ratio("exp_group")
print("stream:", push_stream_srm["reading"])
push_srm = call(push.get_sample_ratio, "exp_group", assignment=push_assignment)
print("table: ", push_srm["reading"], f"(silent paths: {push_srm['silent_paths']})")

On the stream, treatment has about 44 % more users (3,680 against 2,557), a "mismatch": but
opening the app is an *outcome* of the push, so the stream is a post-exposure subset and this
check does not apply to it. On the table, the split is fine. The `UserWarning` is the
assignment id check: fewer than half of the table's ids appear in the stream. It names both
counts so you can tell the two causes apart: most assigned users silent (true here, since a
push audience mostly ignores the push), or ids spelled differently in the table and the stream
(a bug). It is shown once per table, so the later calls on `push_campaign` stay quiet.
Section 3.11 shows the id matching in detail.

## 3. The scorecard: `get_ab_test`

### 3.1 Every metric kind

A scorecard metric is a [path metric](https://retentioneering.com/docs/path-metrics) config,
the dict `get_metrics` takes, plus three optional keys: `name` (the row label), `denominator`
(a second config: a ratio of sums) and `test` (a per-metric test). The kind of metric decides
the default test and how a silent path counts. The lists below are read from the library, so
they are the current registry:

In [ ]:
from retentioneering.tools import ab_test as ab_tools

kinds_table = pd.DataFrame(
    [
        (
            "0/1",
            ", ".join(sorted(ab_tools.BINARY_METRICS)),
            "count as 0",
            "z_test (Wilson / Newcombe)",
        ),
        (
            "total per path",
            ", ".join(sorted(ab_tools.COUNT_METRICS)),
            "count as 0",
            "welch_t_test",
        ),
        (
            "value per path",
            ", ".join(sorted(ab_tools.VALUE_METRICS)),
            "left out of the mean",
            "welch_t_test",
        ),
        (
            "ratio of sums",
            "any total with a denominator",
            "left out (no denominator)",
            "bootstrap",
        ),
    ],
    columns=["kind", "metrics", "silent paths", "default test (test='auto')"],
).set_index("kind")
for name in [
    "TESTS",
    "CORRECTIONS",
    "CORRECTION_SCOPES",
    "PERIOD_BASES",
    "PERIOD_POPULATIONS",
    "BOOTSTRAP_WEIGHTS",
    "VERDICTS",
    "SUPPORT_WARNINGS",
]:
    print(f"{name:>18}: {list(getattr(ab_tools, name))}")
kinds_table

In [ ]:
every_kind = [
    # 0/1 metrics
    CONVERSION,
    {
        "name": "adders",
        "metric": "has_any_event",
        "metric_args": {"events": ["add_to_cart", "quick_add"]},
    },
    {
        "name": "cart_and_checkout",
        "metric": "has_all_events",
        "metric_args": {"events": ["cart", "checkout_start"]},
    },
    {
        "name": "cart_loop",
        "metric": "matches_pattern",
        "metric_args": {"pattern": "cart->cart->cart"},
    },
    {
        "name": "mobile_user",
        "metric": "in_segment",
        "metric_args": {"segment_name": "platform", "segment_level": "mobile"},
    },
    # totals per path
    {"name": "orders_per_user", **ORDERS},
    {"name": "events_per_user", "metric": "length"},
    {"name": "active_days", "metric": "active_days"},
    REVENUE,
    # values per path (seconds for duration and time_between)
    {"name": "path_duration_s", "metric": "duration"},
    {
        "name": "time_to_order_s",
        "metric": "time_between",
        "metric_args": {"start_event": "product_view", "end_event": "order_success"},
    },
    {
        "name": "mean_order_value",
        "metric": "column_mean",
        "metric_args": {"column": "order_value"},
    },
    {
        "name": "last_order_value",
        "metric": "column_last",
        "metric_args": {"column": "order_value"},
    },
    # ratio of sums: revenue / orders
    AOV,
]
scorecard = stream.get_ab_test(
    DIFF, every_kind, assignment=assignment, n_bootstrap=N_BOOT
)
scorecard[
    [
        "metric",
        "population",
        "group1_paths",
        "group1_support",
        "group2_support",
        "conditional",
        "group1_value",
        "group2_value",
        "delta",
        "delta_ci_low",
        "delta_ci_high",
        "p_value_adjusted",
        "verdict",
        "test",
        "ci_basis",
        "p_basis",
        "support_warning",
    ]
]

What to look at, column group by column group (the full list is in the
[`get_ab_test` reference](https://retentioneering.com/docs/tools/get-ab-test)):

- **Who is compared.** `population="assignment"`: every row covers the 10,000 users of the
  table (intention to treat); without `assignment=` it would be `"stream"`. `group1_paths` /
  `group2_paths` are the arms (4,960 and 5,040), `group*_silent_paths` the silent users in each.
- **Who has a value.** `group*_support` and `conditional`. For 0/1 metrics and totals every
  path has a value (silent ones count 0), so `conditional=False`. A value metric has a value
  only where it is defined: `time_to_order_s` and the order-value rows exist for the 1,207 and
  1,196 buyers only, `path_duration_s` (first to last event) for users with events. Those rows are `conditional=True`:
  they compare **the paths with a value**, not the arms, and the treatment may change who has
  one. Read them as descriptions of the buyers, never as the effect on everyone.
- **How it was computed.** `test`, `ci_basis`, `p_basis`: the z-test with Wilson / Newcombe
  intervals for 0/1, Welch for totals and values, a joint path bootstrap for the ratio.

`mean_order_value` (the mean of each buyer's mean order) and `aov` (all revenue over all
orders) are different estimands: the first weighs every buyer equally, the second every order;
they are close here (66.30 and 66.39 in treatment) and part when frequent buyers place orders
of a different size. The
`in_segment` row is a covariate check rather than an outcome: the share of mobile users does
not differ between the arms, as it should not for a variable fixed at signup. `cart_loop` is
the one row with `support_warning="one_arm_only"` (section 3.9).

### 3.2 Reading a row: the `reading` column, `mde`, `rel_delta`

Every row ends with `reading`: one sentence built from the row's own numbers. Here is the
conversion row, an inconclusive one, with the numbers it is built from:

In [ ]:
conversion_row = scorecard.set_index("metric").loc["conversion"]
print(conversion_row["reading"])
print()
print(
    f"delta {pp(conversion_row['delta'])}, interval [{pp(conversion_row['delta_ci_low'])}, {pp(conversion_row['delta_ci_high'])}]"
)
print(
    f"p_value_adjusted {conversion_row['p_value_adjusted']:.3f}, mde {conversion_row['mde'] * 100:.2f} pp at "
    f"mde_alpha {conversion_row['mde_alpha']}, correction_family {conversion_row['correction_family']!r}"
)
print(
    f"rel_delta {conversion_row['rel_delta']:+.4f} = {conversion_row['group1_value']:.4f} / {conversion_row['group2_value']:.4f} - 1, "
    f"interval [{conversion_row['rel_delta_ci_low']:+.4f}, {conversion_row['rel_delta_ci_high']:+.4f}]"
)
print(
    f"each arm: treatment {conversion_row['group1_value']:.4f} [{conversion_row['group1_ci_low']:.4f}, {conversion_row['group1_ci_high']:.4f}], "
    f"control {conversion_row['group2_value']:.4f} [{conversion_row['group2_ci_low']:.4f}, {conversion_row['group2_ci_high']:.4f}]"
)

How to read it:

- **The window** comes first ("over the whole observation period"): a period or horizon row
  names its own window, so a quoted number always carries it.
- **Inconclusive is not "no effect".** The interval, -1.07 to +2.28 pp, is the range of effects
  the data still allow: a drop larger than about 1 pp or a lift larger than about 2.3 pp is
  unlikely; a small effect either way is possible.
- **The `mde`** (2.39 pp) is the effect this design detects with 80 % power at `mde_alpha`: a
  design quantity, not a bound on the true effect. The multiplier ("to detect 1 pp, about
  5.7x the paths are needed in a new test sized before it starts") is the input to sizing the
  *next* test; running this one longer and re-reading it is peeking.
- **`rel_delta`** is group 1 / group 2 - 1: the base is group 2 (control), so +2.5 % means the
  treatment's conversion is 2.5 % above control's, not 2.5 pp.
- **The tail** says what the correction covered: "corrected within this metric only; name one
  decision metric" appears because the scorecard has several metrics and they are not
  corrected against each other (section 3.6).

The `reading` reads the row's numbers only; it does not check the sample ratio or the unit
(sections 2 and 3.12 do).

### 3.3 Metric configs the scorecard refuses

`ABMetricSpec.from_user` validates every entry before anything is computed, with a message that
says how to fix it. There is no per-metric `family` or `exploratory` key: the correction family
is set by `correction_scope` (and reported per row in `correction_family`), and `exploratory`
is a strata column decided by `strata_pre_exposure` (section 3.10).

In [ ]:
bad_entries = {
    "z_test on a total": {**REVENUE, "test": "z_test"},
    "Welch on a ratio": {**AOV, "test": "welch_t_test"},
    "denominator on a 0/1 metric": {**CONVERSION, "denominator": ORDERS},
    "a key the spec does not have": {**CONVERSION, "family": "primary"},
    "metric arguments outside metric_args": {
        "name": "conversion",
        "metric": "has_event",
        "event": "order_success",
    },
    "a bulk metric": {"metric": "has_event_bulk"},
    "a timestamp metric": {"metric": "first_event_time"},
}
for label, entry in bad_entries.items():
    print(f"[{label}]")
    show_error(stream.get_ab_test, DIFF, [entry], assignment=assignment)

### 3.4 `test=`: the test of every metric, and per-metric overrides

`test="auto"` (the default) picks by kind. An explicit `test=` applies to every metric without
its own `test` key; a test that does not fit a metric raises (above). The bootstrap rows carry
`valid_replicates`. On large arms the three tests agree closely, as the conversion rows show.

In [ ]:
two = [CONVERSION, REVENUE]
by_test = []
for test in ["auto", "welch_t_test", "bootstrap"]:
    rows_ = stream.get_ab_test(
        DIFF, two, assignment=assignment, test=test, n_bootstrap=N_BOOT
    )
    by_test.append(rows_.assign(test_argument=test))
only_binary = stream.get_ab_test(
    DIFF, [CONVERSION], assignment=assignment, test="z_test"
)
by_test.append(only_binary.assign(test_argument="z_test"))
per_metric = stream.get_ab_test(
    DIFF,
    [CONVERSION, {**REVENUE, "test": "bootstrap"}, AOV],
    assignment=assignment,
    n_bootstrap=N_BOOT,
).assign(test_argument="auto + per-metric 'test'")
by_test.append(per_metric)
pd.concat(by_test)[
    [
        "test_argument",
        "metric",
        "test",
        "ci_basis",
        "p_basis",
        "delta",
        "delta_ci_low",
        "delta_ci_high",
        "p_value",
        "valid_replicates",
    ]
].reset_index(drop=True)

### 3.5 `correction=`: Holm, Bonferroni, Benjamini-Hochberg, none

The p-values of one **family** are corrected together; the family is the rows of one metric
(its strata and its periods). A plain scorecard has families of one row, where nothing changes,
so the weekly rows of conversion (8 weeks, one family) show the difference. `correction_family`
names the family, `mde_alpha` is the level the `mde` is computed at (alpha / m under a
correction). Intervals stay per comparison; only Bonferroni adds a simultaneous interval
(`delta_ci_low_adj`, `delta_ci_high_adj`) that agrees with its verdict.

In [ ]:
weekly_by_correction = {}
for correction in ["holm", "bonferroni", "bh", "none"]:
    weekly_by_correction[correction] = stream.get_ab_test(
        DIFF, [CONVERSION], assignment=assignment, period="W", correction=correction
    )
weekly = weekly_by_correction["holm"]
comparison = weekly[
    ["period", "delta", "delta_ci_low", "delta_ci_high", "p_value"]
].copy()
for correction, frame in weekly_by_correction.items():
    comparison[f"p_{correction}"] = frame["p_value_adjusted"].to_numpy()
    comparison[f"verdict_{correction}"] = frame["verdict"].to_numpy()
for correction, frame in weekly_by_correction.items():
    print(
        f"{correction:>10}: correction_family={frame['correction_family'].iloc[0]!r}, mde_alpha={frame['mde_alpha'].iloc[0]:.5f}"
    )
comparison

In [ ]:
bonferroni = weekly_by_correction["bonferroni"]
display(
    bonferroni[
        [
            "period",
            "delta_ci_low",
            "delta_ci_high",
            "delta_ci_low_adj",
            "delta_ci_high_adj",
            "verdict",
        ]
    ].head(3)
)
print(bonferroni["reading"].iloc[1])

The second calendar week is an `increase` under every correction. The third week's interval
excludes 0 (+0.33 to +2.22 pp), yet it is `increase` only under BH and no correction: Holm and
Bonferroni read the adjusted p-value (0.057 and 0.065), and the interval is per comparison. That
disagreement is by design; Bonferroni's simultaneous interval for that week (-0.05 to +2.61 pp)
includes 0, in line with its verdict. A rule of thumb: Holm for a few confirmatory rows, BH for
scans over many exploratory ones, Bonferroni when you need intervals that agree with the
verdict.

### 3.6 `correction_scope=`: one family per metric, or one for everything

By default (`"metric"`) each metric is its own family: metrics are **not** corrected against
each other, which is why each reading of a multi-metric scorecard ends "name one decision
metric". `correction_scope="all"` puts every row of the call into one family: for a scorecard
read without a decision metric.

In [ ]:
three = [CONVERSION, REVENUE, AOV]
scopes = pd.concat(
    [
        stream.get_ab_test(
            DIFF,
            three,
            assignment=assignment,
            correction_scope=scope,
            n_bootstrap=N_BOOT,
        ).assign(scope=scope)
        for scope in ["metric", "all"]
    ]
)
display(
    scopes[
        [
            "scope",
            "metric",
            "p_value",
            "p_value_adjusted",
            "correction_family",
            "mde_alpha",
            "mde",
            "verdict",
        ]
    ]
)
print(scopes["reading"].iloc[3])

### 3.7 Guardrails with a harm margin

A guardrail is a metric that must not get worse, with the largest harm you accept declared
before the test. There is no margin argument (a margin-based verdict is deferred): compare the
interval with the margin yourself, and read each guardrail on its own (the default scope), since
correcting guardrails together raises each one's bar for `decrease` and makes harm harder to
detect. A guardrail passes only when its interval clears the margin; a grey guardrail whose
interval reaches past it has **not** passed.

In [ ]:
GUARDRAILS = {
    "revenue_per_user": 1.00,
    "events_per_user": 0.5,
}  # largest acceptable drop, declared up front
guardrail_rows = scorecard.set_index("metric").loc[
    list(GUARDRAILS), ["delta", "delta_ci_low", "delta_ci_high", "verdict"]
]
guardrail_rows["margin"] = [-m for m in GUARDRAILS.values()]
guardrail_rows["passes"] = guardrail_rows["delta_ci_low"] > guardrail_rows["margin"]
guardrail_rows

Revenue per user passes (its lower bound, -0.28, is above -1.00). Events per user does not pass,
although its verdict is not `decrease`: its interval reaches -1.07, past the declared harm of
0.5 events, so the test cannot rule that harm out.

### 3.8 `ci_level=` and `power=`

`ci_level` sets every interval and the verdict's alpha (`1 - ci_level`); `power` only the `mde`.

In [ ]:
levels = []
for kwargs in [{}, {"ci_level": 0.90}, {"ci_level": 0.99}, {"power": 0.9}]:
    row = stream.get_ab_test(DIFF, [CONVERSION], assignment=assignment, **kwargs).iloc[
        0
    ]
    levels.append(
        {
            "arguments": str(kwargs or "defaults"),
            "delta_ci_low": row["delta_ci_low"],
            "delta_ci_high": row["delta_ci_high"],
            "mde_alpha": row["mde_alpha"],
            "mde": row["mde"],
            "verdict": row["verdict"],
        }
    )
pd.DataFrame(levels)

A 90 % interval is narrower, a 99 % one wider; asking for 90 % power raises the `mde` from
2.39 to 2.77 pp: the same test can promise to find only larger effects with more certainty.

### 3.9 `min_support` and every `support_warning`

A row needs enough paths with the behaviour to test it. `min_support` (default 10) is pooled
over both arms: for a 0/1 metric, paths with the event *and* paths without it; for totals,
paths with a non-zero total; for ratios a non-zero numerator; for values a value. Below it the
verdict is `insufficient_data` (estimates kept, no test). `support_warning` flags the rest:

| value | meaning | shown below on |
|---|---|---|
| `"one_arm_only"` | the behaviour exists in one arm only; tested | `quick_add` and the cart loop, Northwind |
| `"no_data"` | no path of either arm has the behaviour (or a value metric has no value in one arm) | the silent users' stratum |
| `"low_support"` | a bootstrap row with fewer than 30 supporting paths in an arm: coverage below nominal | a 200-user draw |
| `"invalid_replicates"` | more than 1 % of bootstrap replicates have no value in an arm: no interval, no p-value | a 40-user draw |

**One arm only, and a renamed event.** The treatment replaces most `add_to_cart` events with
its own `quick_add`. A metric on either event alone measures the renaming, not the behaviour;
the union does not:

In [ ]:
renamed = stream.get_ab_test(
    DIFF,
    [
        {
            "name": "add_to_cart_users",
            "metric": "has_event",
            "metric_args": {"event": "add_to_cart"},
        },
        {
            "name": "quick_add_users",
            "metric": "has_event",
            "metric_args": {"event": "quick_add"},
        },
        {
            "name": "adders",
            "metric": "has_any_event",
            "metric_args": {"events": ["add_to_cart", "quick_add"]},
        },
    ],
    assignment=assignment,
)
renamed[CORE + ["support_warning"]]

In [ ]:
# no_data: the silent users of the table have no platform, so their stratum has no event at all
platform_strata = stream.get_ab_test(
    DIFF,
    [CONVERSION],
    assignment=assignment,
    strata_col="platform",
    strata_pre_exposure=True,
)
print(
    platform_strata.iloc[0][
        [
            "stratum",
            "group1_paths",
            "group2_paths",
            "verdict",
            "support_warning",
            "reading",
        ]
    ].to_dict()
)

# low_support: a 200-user draw has about 20-30 buyers per arm, too few for percentile intervals
small = datasets.load_ab_test(n_users=200, seed=1)
small_assignment = datasets.load_ab_test_assignment(n_users=200, seed=1)
low = small.get_ab_test(
    DIFF, [CONVERSION, AOV], assignment=small_assignment, n_bootstrap=N_BOOT
)

# insufficient_data and invalid_replicates: a 40-user draw, with the default min_support and with 1
tiny = datasets.load_ab_test(n_users=40, seed=3)
tiny_assignment = datasets.load_ab_test_assignment(n_users=40, seed=3)
tiny_default = tiny.get_ab_test(
    DIFF, [CONVERSION, AOV], assignment=tiny_assignment, n_bootstrap=N_BOOT
)
tiny_min1 = tiny.get_ab_test(
    DIFF,
    [CONVERSION, AOV],
    assignment=tiny_assignment,
    n_bootstrap=N_BOOT,
    min_support=1,
)
support_cols = [
    "metric",
    "group1_paths",
    "group2_paths",
    "group1_support",
    "group2_support",
    "delta",
    "delta_ci_low",
    "delta_ci_high",
    "p_value",
    "verdict",
    "valid_replicates",
    "support_warning",
]
display(
    pd.concat(
        [
            low.assign(draw="200 users"),
            tiny_default.assign(draw="40 users, min_support=10"),
            tiny_min1.assign(draw="40 users, min_support=1"),
        ]
    )[["draw"] + support_cols]
)
for _, row in pd.concat([low, tiny_default, tiny_min1]).iterrows():
    print(f"{row['metric']:>10}: {row['reading']}")

Reading the four cases:

- `quick_add_users` exists in treatment only: the row is tested (an `increase` of about
  51 pp) and flagged `one_arm_only`. It is a behaviour the treatment introduced, not an effect
  on an existing one. `add_to_cart_users` shows a large, meaningless drop; `adders`, the union,
  is the behaviour, and it is inconclusive.
- The `<MISSING>` platform stratum (the 267 silent users) is `insufficient_data` with
  `no_data`: "no data in either arm".
- In the 200-user draw the AOV row is a bootstrap row on 31 and 19 buyers: its interval
  under-covers, so it is flagged `low_support`.
- In the 40-user draw, conversion has 7 buyers pooled, below `min_support=10`: both rows are
  `insufficient_data`, with the estimates kept "descriptively". With `min_support=1`
  conversion is tested (and inconclusive), while AOV, on 4 and 3 buyers, stays without an
  interval or a p-value: more than 1 % of its replicates had no buyer in an arm
  (`invalid_replicates`; `valid_replicates` counts the usable ones).

### 3.10 Strata: `strata_col=` and `strata_pre_exposure=`

`strata_col=` repeats every metric within the levels of a segment that is constant per path
(or a column of the assignment table, used first, so silent users keep their stratum). A split
is a randomised comparison only if the treatment could not have changed who is in which level:
the stratum must be fixed **before exposure**. The library records how it knows that, in
`strata_pre_exposure` and `strata_pre_exposure_source`:

- `"declared"`: you said so, with `strata_pre_exposure=True` or
  `add_segment(..., pre_exposure=True)`;
- `"recipe"`: the stream's recipe shows the labels were computed after a `truncate_paths` that
  ends paths at an event (a cut the library sees, not one it can place relative to exposure);
- `"not_pre_exposure"`: neither, with a `UserWarning`; the rows are `exploratory=True`.

Each case on Northwind, where `platform` is fixed at signup:

In [ ]:
strata_cols = [
    "stratum",
    "group1_paths",
    "group2_paths",
    "group1_share",
    "group2_share",
    "delta",
    "delta_ci_low",
    "delta_ci_high",
    "p_value_adjusted",
    "verdict",
    "stratum_vs_rest_p_value",
    "strata_pre_exposure",
    "strata_pre_exposure_source",
    "exploratory",
]
print("declared (strata_pre_exposure=True):")
display(platform_strata[strata_cols])
print("not declared:")
undeclared = call(
    stream.get_ab_test, DIFF, [CONVERSION], assignment=assignment, strata_col="platform"
)
display(undeclared[strata_cols])
first = platform_strata.iloc[0]
print(
    f"balance_p_value {first['balance_p_value']:.3f}, heterogeneity_p_value {first['heterogeneity_p_value']:.3f} "
    f"(dof {first['heterogeneity_dof']}, strata used {first['heterogeneity_strata_used']}), "
    f"correction_family {first['correction_family']!r}"
)
print(undeclared["reading"].iloc[1])

The strata columns:

- `group1_share`, `group2_share`, `balance_p_value`: each stratum's share within each arm and a
  chi-square test of stratum by arm. A share-balance check only: a post-exposure stratum can be
  balanced and still selected by the treatment (below).
- `heterogeneity_p_value` (Cochran's Q over the strata that pass a support guard, with
  `heterogeneity_dof` and `heterogeneity_strata_used`): do the effects differ between strata at
  all? Read it before any single stratum. Here p 0.94: no sign that platforms differ.
- `stratum_vs_rest_p_value`: this stratum's delta against all the others pooled. Use it, not
  "significant here, not there", to say a stratum is different.
- The strata of one metric are one correction family (`metric=conversion/strata`).
- Undeclared, the same rows are flagged `exploratory` and their reading says so.

**`"recipe"` and a post-exposure split.** Length bins computed on the window before the first
`product_view` (the exposure point) and carried back with `path_values` read as `"recipe"`.
The same kind of bins computed on the whole path, after exposure, warn and are flagged:

In [ ]:
pre_exposure = stream.truncate_paths(
    "path_start", [{"pattern": "product_view", "offset": -1}, "path_end"]
)
window_bins = {
    "metric": {"metric": "length"},
    "edges": [3],
    "segment_levels": ["short", "long"],
}
window_labels = pre_exposure.add_segment(
    "window_len", metric_bins=window_bins
).get_path_segments("window_len")
print("labels computed after a truncation:", window_labels.attrs)
recipe_strata = stream.add_segment("window_len", path_values=window_labels).get_ab_test(
    DIFF, [CONVERSION], assignment=assignment, strata_col="window_len"
)
display(recipe_strata[strata_cols])

whole_path_bins = {
    "metric": {"metric": "length"},
    "quantiles": 2,
    "segment_levels": ["short", "long"],
}
post_strata = call(
    stream.add_segment("len_bin", metric_bins=whole_path_bins).get_ab_test,
    DIFF,
    [CONVERSION],
    assignment=assignment,
    strata_col="len_bin",
)
display(post_strata[strata_cols + ["balance_p_value"]])

The first split (cut at the first `product_view`) reads `"recipe"`, `exploratory=False`. The
second, on the whole path, warns and is `not_pre_exposure` / `exploratory=True`, with a balance
p-value that raises no alarm (0.33): balance is not a guard against post-exposure strata, the
way the stratum was computed is. Whether the cut sits before exposure is your argument (the
library sees a cut, not where exposure starts), so for a cut you rely on, declare it
(section 10).

**Strata from the assignment table.** `push_campaign`'s table carries `platform`. A column of
the table is used first, so the silent users (most of this audience) keep their platform
instead of falling into `<MISSING>`. A table column is not declared either, so declare it:

In [ ]:
push_strata = call(
    push.get_ab_test,
    DIFF,
    [{"name": "purchase", "metric": "has_event", "metric_args": {"event": "purchase"}}],
    assignment=push_assignment,
    strata_col="platform",
    strata_pre_exposure=True,
)
push_strata[
    [
        "stratum",
        "group1_paths",
        "group2_paths",
        "group1_silent_paths",
        "group2_silent_paths",
        "delta",
        "delta_ci_low",
        "delta_ci_high",
        "verdict",
        "strata_pre_exposure_source",
    ]
]

### 3.11 Who is compared: `diff` forms, `population` and id matching

`diff` names the two groups, resolved **per path**:

- `("segment_col", group1, group2)`: a segment of the stream;
- with `assignment=`, `("arm", group1, group2)` reads the table alone, and a segment name adds
  the check that every path's segment agrees with the table;
- `(ids1, ids2)`: two lists of path ids (of the stream: silent ids raise; pass a table for
  them).

The assignment table can be a DataFrame with `arm` (and optionally `assigned_at`) or a Series
`path_id -> arm`. Its ids are matched to the stream's: as they are, or by their string form
when none matches (ids that went through a CSV or JSON); none matching raises, fewer than half
matching warns.

In [ ]:
seen = set(stream.df["user_id"])
ids_treatment = [
    i for i in assignment.index[assignment["arm"] == "treatment"] if i in seen
]
ids_control = [i for i in assignment.index[assignment["arm"] == "control"] if i in seen]
as_strings = assignment.copy()
as_strings.index = as_strings.index.astype(str)  # e.g. read back from a CSV

forms = {
    "segment, no table": dict(diff=DIFF),
    "segment + table": dict(diff=DIFF, assignment=assignment),
    "'arm' + table": dict(diff=("arm", "treatment", "control"), assignment=assignment),
    "Series table": dict(diff=DIFF, assignment=assignment["arm"]),
    "table with string ids": dict(diff=DIFF, assignment=as_strings),
    "id lists": dict(diff=(ids_treatment, ids_control)),
}
form_rows = []
for label, kwargs in forms.items():
    diff = kwargs.pop("diff")
    row = call(stream.get_ab_test, diff, [CONVERSION], **kwargs).iloc[0]
    form_rows.append(
        {
            "form": label,
            **row[
                [
                    "population",
                    "group1_paths",
                    "group2_paths",
                    "unassigned_paths",
                    "delta",
                    "delta_ci_low",
                    "delta_ci_high",
                ]
            ].to_dict(),
            "silent": row.get("group1_silent_paths", np.nan),
        }
    )
pd.DataFrame(form_rows).set_index("form")

In [ ]:
# What the library refuses
show_error(stream.get_ab_test, ("exp_group", "treatment", "treatment"), [CONVERSION])
show_error(
    stream.get_ab_test,
    ("exp_group", "treatment", "treatmnt"),
    [CONVERSION],
    assignment=assignment,
)
all_treatment = list(assignment.index[assignment["arm"] == "treatment"])
show_error(stream.get_ab_test, (all_treatment, ids_control), [CONVERSION])
foreign = assignment.set_axis(assignment.index + 10_000_000)
show_error(stream.get_ab_test, DIFF, [CONVERSION], assignment=foreign)

With the table, `population="assignment"` and the silent users are counted (132 in treatment);
without it, `population="stream"`: 9,733 users, the silent ones dropped. The string-id table
and the Series give the same row as the table. Id lists of the stream reproduce the stream
read; an id list that names silent users raises, with the advice to pass a table instead. A
table that matches no path raises; one that matches fewer than half warns (section 2.4).

### 3.12 `path_col=` and the unit guard

Every interval treats one path as one independent unit, so `path_col` must be the unit that was
randomised. Northwind assigned users; asking for sessions raises, with a table and without one
(the arm is constant within every user, so the library reads `user_id` as the unit):

In [ ]:
try:
    stream.get_ab_test(DIFF, [CONVERSION], path_col="session_id")
except Exception as error:
    print(f"{type(error).__name__}: {error}")
try:
    stream.get_ab_test(DIFF, [CONVERSION], assignment=assignment, path_col="session_id")
except Exception as error:
    print(f"{type(error).__name__}: {error}")

**`session_assignment`** is the other direction: sessions were randomised. At the user grain the
scorecard refuses, both from the stream's segment (users fall in both groups) and from the
session table (a finer unit than the paths):

In [ ]:
ADD = {
    "name": "add_rate",
    "metric": "has_event",
    "metric_args": {"event": "add_to_cart"},
}
show_error(sessions.get_ab_test, DIFF, [ADD])
show_error(sessions.get_ab_test, DIFF, [ADD], assignment=session_table)
per_session_card = sessions.get_ab_test(
    DIFF, [ADD], assignment=session_table, path_col="session_id"
)
per_session_card[CORE + ["group1_paths", "group2_paths", "delta_se"]]

At the session grain the scorecard runs and reads a large increase, but its interval is too
narrow: sessions of one user share the user's habits (and here a per-user effect), and a few
heavy users contribute hundreds of sessions each. `get_ab_paths` (section 5) gives the
per-session table, on which a user-clustered standard error is a few lines; the clean
user-level read is each user's first session, cut with `truncate_paths`:

In [ ]:
session_paths = sessions.get_ab_paths(
    DIFF, [ADD], assignment=session_table, path_col="session_id"
)
user_of = session_paths.index.str.rsplit("_", n=1).str[0]
y = session_paths["add_rate"].to_numpy()
X = np.column_stack(
    [np.ones(len(y)), (session_paths["arm"] == "treatment").to_numpy(float)]
)
bread = np.linalg.inv(X.T @ X)
residual = y - X @ (bread @ X.T @ y)
scores = (
    pd.DataFrame(X * residual[:, None]).groupby(user_of.to_numpy()).sum().to_numpy()
)
n_users_ = len(scores)
clustered_se = float(
    np.sqrt((bread @ scores.T @ scores @ bread)[1, 1] * n_users_ / (n_users_ - 1))
)
naive_se = per_session_card["delta_se"].iloc[0]
print(
    f"session-grain delta_se {naive_se:.4f}; user-clustered SE {clustered_se:.4f} ({clustered_se / naive_se:.2f}x)"
)

first_sessions = sessions.truncate_paths(
    "path_start",
    [
        {"pattern": "session_start->.*->session_start", "at": 1, "offset": -1},
        "path_end",
    ],
)
print("first sessions:", first_sessions.get_sample_ratio("exp_group")["reading"])
first_session_card = first_sessions.get_ab_test(DIFF, [ADD])
planted_effect = (
    datasets.load_ab_test("session_assignment", as_dataframe=True, with_truth=True)
    .groupby("user_id")["planted_effect"]
    .first()
)
print(
    f"planted per-user effect: mean {pp(planted_effect.mean())} before clipping (the realised mean is about +7 pp)"
)
first_session_card[CORE + ["group1_paths", "group2_paths", "population"]]

The clustered standard error is about 1.55 times the session-grain `delta_se`: the reported
interval was far too narrow. The first-session cut has one arm per user (the sample ratio check
passes with no multi-valued paths) and reads +7.3 pp, close to the planted +7 pp. The right
conclusion of this dataset is to fix the assignment (per user) and rerun; until then, report
the first-session read with its caveat, never the session-grain interval.

## 4. Time: periods, horizon, novelty

### 4.1 Calendar periods and `period_population`

`period=` gives one row per period: a calendar unit (`"D"`, `"W"`, `"M"`, `"Q"`, `"Y"`, also
`"W-MON"` etc.) or a fixed length (`"14D"`, `"12h"`). With `period_basis="calendar"` (the
default) and `period_population="itt"` (the default), a period's row holds every user assigned
by the end of the period, active in it or not; the periods of one metric are one correction
family. `period_population="active"` keeps the users active in the period: survivors the
treatment may have selected, so the rows are `descriptive`, with no p-value.

In [ ]:
period_cols = [
    "period",
    "period_start",
    "period_end",
    "period_complete",
    "population",
    "group1_paths",
    "group2_paths",
    "delta",
    "delta_ci_low",
    "delta_ci_high",
    "p_value_adjusted",
    "verdict",
]
weekly_active = stream.get_ab_test(
    DIFF, [CONVERSION], assignment=assignment, period="W", period_population="active"
)
monthly = stream.get_ab_test(DIFF, [CONVERSION], assignment=assignment, period="M")
fortnights = stream.get_ab_test(DIFF, [CONVERSION], assignment=assignment, period="14D")
display(
    pd.concat([weekly.assign(rows="W, itt"), weekly_active.assign(rows="W, active")])[
        ["rows"] + period_cols
    ]
)
display(
    pd.concat([monthly.assign(rows="M"), fortnights.assign(rows="14D")])[
        ["rows"] + period_cols
    ]
)
print(weekly["reading"].iloc[0])
print(weekly_active["reading"].iloc[0])
show_error(stream.get_ab_test, DIFF, [CONVERSION], assignment=assignment, period=7)

The weekly ITT delta drifts from positive in the first weeks to negative in the last ones. The
last week's `period_complete` is False (the data end before the week does). Calendar weeks mix
users of every exposure age, so a drift across them looks like novelty without measuring it:
the next two sections measure it.

### 4.2 `horizon=`: the same window for every user

The open-ended scorecard counts everything a user did until the data end: eight weeks for an
early user, two for a late one. `horizon="28D"` counts only events in
`[assigned_at, assigned_at + 28 days)` and leaves out users whose window had not closed by the
last event (`immature_paths`). It needs `assigned_at`, i.e. an assignment table.

In [ ]:
horizon_rows = []
for horizon in [None, "14D", "28D", "42D"]:
    row = stream.get_ab_test(
        DIFF, [CONVERSION], assignment=assignment, horizon=horizon
    ).iloc[0]
    horizon_rows.append(
        {
            "horizon": horizon or "none",
            **row.reindex(
                [
                    "group1_paths",
                    "group2_paths",
                    "immature_paths",
                    "delta",
                    "delta_ci_low",
                    "delta_ci_high",
                    "p_value",
                    "mde",
                    "verdict",
                    "reading",
                ]
            ).to_dict(),
        }
    )
horizons = pd.DataFrame(horizon_rows).set_index("horizon")
display(horizons.drop(columns="reading"))
print(horizons.loc["28D", "reading"])
show_error(
    stream.get_ab_test, DIFF, [CONVERSION], horizon="28D"
)  # no table, no assigned_at

Different horizons answer different questions: +2.75 pp within 14 days, +2.09 pp within 28,
+1.00 pp (inconclusive) within 42. Choose the horizon that matches the decision **before** the
test; trying horizons until one turns green is peeking.

### 4.3 `period_basis="since_assignment"`: bins by exposure age

Bins of time since each user's own `assigned_at` (`"0-13"`, `"14-27"`, ...). A user counts in a
bin once the bin's window has closed (else `immature_paths`); a bin no user has closed is left
out. With `horizon="42D"`, every bin holds the same users.

In [ ]:
since = stream.get_ab_test(
    DIFF,
    [CONVERSION],
    assignment=assignment,
    period="14D",
    period_basis="since_assignment",
)
since_42 = stream.get_ab_test(
    DIFF,
    [CONVERSION],
    assignment=assignment,
    period="14D",
    period_basis="since_assignment",
    horizon="42D",
)
since_cols = [
    "period",
    "period_complete",
    "group1_paths",
    "group2_paths",
    "immature_paths",
    "delta",
    "delta_ci_low",
    "delta_ci_high",
    "p_value_adjusted",
    "verdict",
]
display(
    pd.concat([since.assign(horizon="none"), since_42.assign(horizon="42D")])[
        ["horizon"] + since_cols
    ]
)
print(since["reading"].iloc[2])

The effect is front-loaded: an increase in days 0-13 (+2.75 pp), a decrease in days 28-41
(-1.48 pp, adjusted p 0.023). Without a horizon each bin holds the users whose window closed
(9,995, then 8,702, then 6,336 users) and `period_complete` is False wherever some assigned
user's window was still open; with `horizon="42D"` all three bins hold the same 6,336 users.

### 4.4 `get_ab_bootstrap`: joint replicates, contrasts, weights, `"auto"`, `random_state`

Two rows of the table above do not test that the effect *changed*: their estimates are
correlated (same users), so their intervals cannot be combined by eye. `get_ab_bootstrap`
resamples users once and evaluates every metric, stratum and period on the same replicates, and
returns them as a long table. `contrast={"name": (periods_a, periods_b)}` adds rows for the
change between periods, on the paths observed in all of them.

In [ ]:
replicates = stream.get_ab_bootstrap(
    DIFF,
    [CONVERSION],
    assignment=assignment,
    period="14D",
    period_basis="since_assignment",
    contrast={"novelty": (["0-13"], ["28-41"])},
    n_bootstrap=N_BOOT,
)
print("columns:", list(replicates.columns), "| rows:", len(replicates))
display(replicates.head(3))
novelty = replicates.loc[replicates["contrast"] == "novelty", "delta"]
novelty_point = since_42.set_index("period")["delta"]
novelty_point = novelty_point["0-13"] - novelty_point["28-41"]
novelty_low, novelty_high = novelty.quantile([0.025, 0.975])
print(
    f"novelty contrast: {pp(novelty_point)}, 95% interval [{pp(novelty_low)}, {pp(novelty_high)}] "
    f"from {len(novelty)} replicates; share of replicates above 0: {(novelty > 0).mean():.2f}"
)

For the 6,336 users observed in both windows, the effect in days 0-13 exceeds the effect in
days 28-41 by 4.86 pp, with an interval that excludes 0. The point estimate is the difference
of the two `horizon="42D"` rows; the replicates give the interval (their mean is not the
estimate). Novelty is one explanation; users buying early what they would have bought later is
another, and the data do not separate them.

The replicates are the ones `get_ab_test` uses for its bootstrap rows (same seed, same engine),
so custom summaries agree with the scorecard. The remaining bootstrap arguments:

- `bootstrap_weights="multinomial"` (default) resamples each arm's paths with replacement;
  `"poisson"` gives each path a Poisson(1) weight;
- `n_bootstrap="auto"` = `max(1000, ceil(20 * m / alpha))`, `m` the largest family: enough
  replicates for the smallest adjusted p-value the family can need;
- `random_state` (default 42) fixes the draw: the same seed gives the same interval.

In [ ]:
boot_rows = []
for label, kwargs in [
    ("multinomial, seed 42", dict(n_bootstrap=N_BOOT)),
    ("poisson, seed 42", dict(n_bootstrap=N_BOOT, bootstrap_weights="poisson")),
    ("multinomial, seed 7", dict(n_bootstrap=N_BOOT, random_state=7)),
    ("multinomial, seed 7 again", dict(n_bootstrap=N_BOOT, random_state=7)),
    ("n_bootstrap='auto'", dict(n_bootstrap="auto")),
]:
    row = stream.get_ab_test(DIFF, [AOV], assignment=assignment, **kwargs).iloc[0]
    boot_rows.append(
        {
            "call": label,
            **row[
                [
                    "delta",
                    "delta_ci_low",
                    "delta_ci_high",
                    "p_value",
                    "valid_replicates",
                ]
            ].to_dict(),
        }
    )
display(pd.DataFrame(boot_rows).set_index("call"))

auto_strata = stream.get_ab_test(
    DIFF,
    [AOV],
    assignment=assignment,
    n_bootstrap="auto",
    strata_col="platform",
    strata_pre_exposure=True,
)
print("'auto' with a family of 3 strata:", auto_strata["valid_replicates"].tolist())

aov_reps = stream.get_ab_bootstrap(
    DIFF, [AOV], assignment=assignment, n_bootstrap=N_BOOT
)
aov_card = stream.get_ab_test(
    DIFF, [AOV], assignment=assignment, n_bootstrap=N_BOOT
).iloc[0]
print(
    f"replicate percentiles {aov_reps['delta'].quantile([0.025, 0.975]).round(4).tolist()} = scorecard interval "
    f"[{aov_card['delta_ci_low']:.4f}, {aov_card['delta_ci_high']:.4f}]; "
    f"P(treatment AOV higher) = {(aov_reps['delta'] > 0).mean():.2f}"
)
show_error(stream.get_ab_test, DIFF, [AOV], assignment=assignment, n_bootstrap=50)

`"auto"` draws 1,000 replicates for a family of one row and 1,200 for the three platform strata
(alpha 0.05, m = 3); the `<MISSING>` stratum has no buyer, so none of its replicates is valid.
Fewer than 100 replicates are refused.

Because every stratum is evaluated on the same replicates, `get_ab_bootstrap(strata_col=...)`
gives a joint interval for any comparison between strata, for example "is the effect on mobile
larger than on desktop?". (`stratum_vs_rest_p_value` in the scorecard is the closed-form version
of the same question.) Here the interval includes 0, in line with the heterogeneity p-value of
section 3.10: no sign that the platforms react differently.

In [ ]:
strata_reps = stream.get_ab_bootstrap(
    DIFF,
    [CONVERSION],
    assignment=assignment,
    strata_col="platform",
    strata_pre_exposure=True,
    n_bootstrap=N_BOOT,
)
by_stratum = strata_reps.pivot_table(
    index="replicate", columns="stratum", values="delta"
)
mobile_minus_desktop = by_stratum["mobile"] - by_stratum["desktop"]
point = platform_strata.set_index("stratum")["delta"]
print(f"columns: {list(strata_reps.columns)}")
print(
    f"effect on mobile minus effect on desktop: {pp(point['mobile'] - point['desktop'])}, 95% interval "
    f"[{pp(mobile_minus_desktop.quantile(0.025))}, {pp(mobile_minus_desktop.quantile(0.975))}]"
)

### 4.5 `trial_conversion`: a test read too early

A 14-day trial and an early paywall offer on day 3. The data end three days after the last
trial started, and users subscribe up to day 20.

In [ ]:
SUBSCRIBED = {
    "name": "subscribed",
    "metric": "has_event",
    "metric_args": {"event": "subscribe"},
}
trial_open = trial.get_ab_test(DIFF, [SUBSCRIBED], assignment=trial_assignment)
trial_21 = trial.get_ab_test(
    DIFF, [SUBSCRIBED], assignment=trial_assignment, horizon="21D"
)
display(
    pd.concat(
        [trial_open.assign(read="whole window"), trial_21.assign(read="horizon 21D")]
    )[["read", "group1_paths", "group2_paths", "immature_paths"] + CORE]
)
print(trial_21["reading"].iloc[0])
trial_bins = trial.get_ab_test(
    DIFF,
    [SUBSCRIBED],
    assignment=trial_assignment,
    period="7D",
    period_basis="since_assignment",
    horizon="21D",
)
display(trial_bins[since_cols])
trial_reps = trial.get_ab_bootstrap(
    DIFF,
    [SUBSCRIBED],
    assignment=trial_assignment,
    period="7D",
    period_basis="since_assignment",
    horizon="21D",
    contrast={"earlier": (["0-6"], ["14-20"])},
    n_bootstrap=N_BOOT,
)
shift = trial_reps.loc[trial_reps["contrast"] == "earlier", "delta"].quantile(
    [0.025, 0.975]
)
print(
    f"contrast days 0-6 minus days 14-20: interval [{pp(shift.iloc[0])}, {pp(shift.iloc[1])}]"
)

trial_truth = (
    datasets.load_ab_test("trial_conversion", as_dataframe=True, with_truth=True)
    .groupby("user_id")
    .agg(arm=("exp_group", "first"), day=("planted_subscribe_day", "first"))
)
trial_truth.assign(subscribes=trial_truth["day"].notna()).groupby("arm")[
    "subscribes"
].mean().rename("planted share of subscribers among users with events").to_frame()

The whole-window read says `increase` (+3.3 pp, p < 0.001), and it is an artefact: a user who
started the trial ten days before the data end has had the chance to take the day-3 offer but
not yet to reach the end of the trial, so treatment conversions are counted and control ones
are not yet. With `horizon="21D"` (trial plus grace week) every user has the same window and
4,410 immature users are left out: +0.9 pp, inconclusive. The since-assignment bins show the
mechanism: subscriptions move from days 14-20 to days 0-6, with a contrast far from 0. The
planted truth agrees: about 18 % of users subscribe in both arms.

### 4.6 `aa_calibration`: an A/A test

Both arms run the same build, so every ITT read is a true null; daily rows show what a clean
null looks like (and that one A/A run is a smoke test, not a calibration of the false-positive
rate, which needs many re-randomised splits: an A/A replay the library does not offer yet).

In [ ]:
aa_metrics = [
    CONVERSION,
    {"name": "orders_per_user", **ORDERS},
    REVENUE,
    AOV,
    {**REVENUE, "name": "revenue_bootstrap", "test": "bootstrap"},
]
aa_card = aa.get_ab_test(DIFF, aa_metrics, assignment=aa_assignment, n_bootstrap=N_BOOT)
display(aa_card[CORE + ["test"]])
aa_daily = aa.get_ab_test(DIFF, [CONVERSION], assignment=aa_assignment, period="D")
print(
    f"daily conversion rows: {len(aa_daily)}, verdicts {aa_daily['verdict'].value_counts().to_dict()}, "
    f"raw p < 0.05 in {int((aa_daily['p_value'] < 0.05).sum())}"
)
aa_paths = aa.get_ab_paths(DIFF, [REVENUE], assignment=aa_assignment)
top_revenue = aa_paths.sort_values("revenue_per_user", ascending=False).head(8)
whales = (
    datasets.load_ab_test("aa_calibration", as_dataframe=True, with_truth=True)
    .query("whale")["user_id"]
    .unique()
)
print(
    "the six largest paths by revenue are the planted whales:",
    set(top_revenue.index[:6]) == set(whales),
    "| their arm:",
    top_revenue["arm"].iloc[:6].unique().tolist(),
)
top_revenue

Conversion, orders and every daily row are inconclusive, as a null should be. Revenue per user
and AOV read `increase`, and so does revenue under the bootstrap: the six largest paths are six
whale accounts of one reseller, created together and assigned together to treatment. The
bootstrap does not help, since it resamples the same six paths. A robust metric does:
winsorise the order value at its 99th percentile in a custom column and declare that metric.

In [ ]:
aa_events = datasets.load_ab_test("aa_calibration", as_dataframe=True)
cap = aa_events["order_value"].quantile(0.99)
aa_events["order_value_capped"] = aa_events["order_value"].clip(upper=cap)
aa_capped = rete.Eventstream(
    aa_events,
    {
        "path_cols": ["user_id", "session_id"],
        "segment_cols": ["exp_group", "platform"],
        "custom_cols": ["order_value", "order_value_capped"],
    },
)
capped_metrics = [
    {
        "name": "revenue_capped",
        "metric": "column_sum",
        "metric_args": {"column": "order_value_capped"},
    },
    {
        "name": "aov_capped",
        "metric": "column_sum",
        "metric_args": {"column": "order_value_capped"},
        "denominator": ORDERS,
    },
]
print(f"cap: {cap:.2f}")
aa_capped.get_ab_test(
    DIFF, capped_metrics, assignment=aa_assignment, n_bootstrap=N_BOOT
)[CORE + ["test"]]

Both capped rows are inconclusive. The lesson is about the assignment, not the metric alone:
accounts created together were assigned together, so the reseller is the unit they share.

### 4.7 `push_campaign`: intention to treat over the table

The push changes who opens the app, so the stream (users who showed up) is selected by the
treatment. Only the table is the experiment: silent users count as not converted, events before
`assigned_at` are dropped.

In [ ]:
PURCHASE = {
    "name": "purchase",
    "metric": "has_event",
    "metric_args": {"event": "purchase"},
}
OPENED = {
    "name": "opened_app",
    "metric": "has_event",
    "metric_args": {"event": "app_open"},
}
push_itt = call(push.get_ab_test, DIFF, [PURCHASE, OPENED], assignment=push_assignment)
push_naive = push.get_ab_test(DIFF, [PURCHASE, OPENED])
before = push.df["timestamp"] < push.df["user_id"].map(push_assignment["assigned_at"])
print(
    f"events before assigned_at: {int(before.sum())} from {push.df.loc[before, 'user_id'].nunique()} users (dropped by assignment=)"
)
display(
    pd.concat([push_itt, push_naive])[
        [
            "metric",
            "population",
            "group1_paths",
            "group2_paths",
            "group1_silent_paths",
            "group2_silent_paths",
        ]
        + CORE[1:]
    ]
)
compliance = (
    datasets.load_ab_test("push_campaign", as_dataframe=True, with_truth=True)
    .groupby("user_id")["compliance"]
    .first()
)
pd.crosstab(compliance, push_assignment["arm"].reindex(compliance.index), margins=True)

ITT over the table: purchases rise by +1.94 pp of the audience (the planted effect is about
+1.4 pp; +1.9 pp at this seed), and more treatment users open the app. The stream-only read
has the opposite sign (-2.27 pp, `decrease`): it compares the users who showed up, and the
users the push brought in buy less than those who open anyway. Its `opened_app` row is
`insufficient_data`: on the stream, everyone opened the app. The truth column confirms the
mechanism: `push_only` users appear in the log only in treatment.

## 5. The per-path table: `get_ab_paths`

`get_ab_paths` returns what the scorecard aggregates: one row per compared path with its `arm`,
`assigned_at`, `silent`, `mature` (in the analysed population; False for a path whose
`horizon` window had not closed, with NaN metrics) and one column per metric (a ratio as
`<name>_numerator` and `<name>_denominator`). Use it for what the scorecard does not do:
variance reduction, rank or quantile tests, custom plots, clustered errors (section 3.12).

In [ ]:
paths = stream.get_ab_paths(
    DIFF, [CONVERSION, REVENUE, AOV], assignment=assignment, horizon="28D"
)
print("index:", paths.index.name, "| columns:", list(paths.columns))
display(paths.groupby("arm").head(3))
mature = paths[paths["mature"]]
by_arm = mature.groupby("arm").agg(
    conversion=("conversion", "mean"),
    revenue_per_user=("revenue_per_user", "mean"),
    aov_numerator=("aov_numerator", "sum"),
    aov_denominator=("aov_denominator", "sum"),
)
by_arm["aov"] = by_arm["aov_numerator"] / by_arm["aov_denominator"]
card_28 = stream.get_ab_test(
    DIFF,
    [CONVERSION, REVENUE, AOV],
    assignment=assignment,
    horizon="28D",
    n_bootstrap=N_BOOT,
)
joined = card_28.set_index("metric")[["group1_value", "group2_value"]].join(
    by_arm.T.rename(
        columns={"treatment": "treatment_from_paths", "control": "control_from_paths"}
    )
)
print(
    f"immature (not mature) paths: {int((~paths['mature']).sum())} = scorecard immature_paths {card_28['immature_paths'].iloc[0]}"
)
joined

In [ ]:
# Something the scorecard does not run: a rank test of revenue among buyers in the first 28 days
buyers = mature[mature["aov_denominator"] > 0]
rank_test = stats.mannwhitneyu(
    buyers.loc[buyers["arm"] == "treatment", "revenue_per_user"],
    buyers.loc[buyers["arm"] == "control", "revenue_per_user"],
)
print(
    f"buyers: {buyers['arm'].value_counts().to_dict()}; Mann-Whitney U p = {rank_test.pvalue:.3f} "
    "(conditional on buying: a description of buyers, not an ITT effect)"
)

# And a few raw paths per arm, to keep the numbers honest about what users do
for arm in ["treatment", "control"]:
    sample = (
        stream.filter_paths(
            [
                {"in_segment": {"segment_name": "exp_group", "segment_level": arm}},
                {"has_event": "order_success"},
            ]
        )
        .sample_paths(n=2, random_state=7)
        .to_dataframe()
    )
    for user_id, user_events in sample.groupby("user_id", sort=False):
        print(f"{arm} user {user_id}:")
        for _, session in user_events.groupby("session_id", sort=False):
            print("   ", " -> ".join(session["event"]))

The per-arm aggregates of the mature paths are exactly the scorecard's `group1_value` /
`group2_value`: the mean of a 0/1 metric or a total, and summed numerator over summed
denominator for the ratio. The immature paths stay in the table (`mature=False`) so you can see
who was left out. Section 3.12 used the same table for a user-clustered standard error.

## 6. Funnels with intervals

`funnel_data(steps, diff=, ci_level=, correction=)` adds Wilson intervals to every rate and, in
diff mode, a Newcombe interval to every delta and a pooled z-test p-value to every **overall**
step (the share of the arm's paths that reached it), corrected across the steps. The
step-to-step (conditional) rates get an interval but no p-value: their denominator, the users
who reached the previous step, is chosen by the treatment. `funnel(...)` draws the same numbers.

In [ ]:
steps = ["product_view", "cart", "checkout_start", "payment", "order_success"]
funnel = stream.funnel_data(steps, diff=DIFF, ci_level=0.95, correction="holm")
print({key: value for key, value in funnel.items() if key != "steps"})
funnel_table = pd.DataFrame(funnel["steps"]).set_index("step")
# relative delta, base = group 2 (control), as rel_delta in the scorecard
funnel_table["rel_delta_conversion_rate"] = (
    funnel_table["funnel1_conversion_rate"] / funnel_table["funnel2_conversion_rate"]
    - 1
)
funnel_table[
    [
        "funnel1_conversion_rate",
        "funnel2_conversion_rate",
        "delta_conversion_rate",
        "rel_delta_conversion_rate",
        "delta_conversion_rate_ci_low",
        "delta_conversion_rate_ci_high",
        "delta_conversion_rate_p_value_adjusted",
        "step_conditional",
        "delta_step_conversion_rate",
        "delta_step_conversion_rate_ci_low",
        "delta_step_conversion_rate_ci_high",
    ]
]

In [ ]:
stream.funnel(steps, diff=DIFF, ci_level=0.95, correction="holm").render_static()

How to read it:

- **Overall steps** (`delta_conversion_rate`, out of every path of the arm) are randomised
  comparisons: all grey after Holm (`correction_family` `metric=delta_conversion_rate/steps`).
- **The relative delta** divides by group 2: a step reached by 25.0 % of treatment paths and
  24.4 % of control paths is +2.5 % relative (`funnel1 / funnel2 - 1`), +0.62 pp absolute.
- **Conditional steps** (`step_conditional=True`) have an interval and no p-value. Two of them
  exclude 0, in opposite directions (`cart -> checkout_start` up, `checkout_start -> payment`
  down): a hint at a mechanism (the treatment's "checkout now" shortcut), not an effect.
- **The denominator is the paths with events** (`funnel1_total_paths`), not the assignment
  table: for an ITT read of a step use `get_ab_test` with a 0/1 metric of that step and
  `assignment=`.

Other levels and corrections work the same way; a correction without `ci_level` is refused,
and so is a session grain with intervals (the unit guard of section 3.12; a descriptive funnel
without `ci_level` keeps the widgets' event-level split and no guard applies).

In [ ]:
funnel_bh = stream.funnel_data(steps, diff=DIFF, ci_level=0.90, correction="bh")
print(
    pd.DataFrame(funnel_bh["steps"])[
        [
            "step",
            "delta_conversion_rate_ci_low",
            "delta_conversion_rate_ci_high",
            "delta_conversion_rate_p_value",
            "delta_conversion_rate_p_value_adjusted",
        ]
    ]
)
show_error(stream.funnel_data, steps, diff=DIFF, correction="holm")
show_error(stream.funnel_data, steps, diff=DIFF, ci_level=0.95, path_col="session_id")

# get_conversion_rate(diff=): the same idea for one start -> end pair; conditional unless it starts at path_start
pairs = stream.get_conversion_rate(["path_start", "cart"], "order_success", diff=DIFF)
pairs[
    [
        "start_anchor",
        "group1_paths_with_start",
        "group2_paths_with_start",
        "group1_conversion_rate",
        "group2_conversion_rate",
        "delta",
        "delta_ci_low",
        "delta_ci_high",
        "p_value",
        "conditional",
    ]
]

`get_conversion_rate(diff=)` reports intervals and a p-value at the path level and flags every
row whose start anchor is not the path start `conditional=True`: "conversion after reaching the
cart" compares users the treatment selected, so it is a description, like a conditional funnel
step. From `path_start` it is the overall conversion of users with events (`population="stream"`
in scorecard terms).

## 7. Which edges and steps differ: `get_ab_transitions` and `get_ab_steps`

A transition graph in diff mode colours every edge whose weight differs between the arms, noise
included. `get_ab_transitions(diff, edge_weight=...)` tests each edge instead: one row per edge
(`path_start` and `path_end` included) with each arm's value, the delta and its per-edge
interval, a p-value adjusted over the tested edges (Benjamini-Hochberg by default), a `verdict`
and a `reading`. The arms, the population (`assignment=` for ITT), the unit guard, `horizon`,
the pooled support rule and the verdict are the scorecard's. It is an exploratory screen, read
after the scorecard: the family is every edge with enough support, found by looking, so a
discovery is a hypothesis for a declared metric of a new test. `get_ab_steps` does the same for
the cells of the step matrix.

In [ ]:
EDGE_COLS = [
    "source",
    "target",
    "group1_edge_paths",
    "group2_edge_paths",
    "group1_value",
    "group2_value",
    "delta",
    "delta_ci_low",
    "delta_ci_high",
    "p_value_adjusted",
    "verdict",
    "support_warning",
]
edges = stream.get_ab_transitions(DIFF, assignment=assignment)
edge_rows = edges.set_index(["source", "target"])
print(
    f"{edges.attrs['n_tested']} of {len(edges)} edges tested | "
    f"{edges['verdict'].value_counts().to_dict()} | population {edges['population'].iloc[0]} | "
    f"family {edges['correction_family'].iloc[0]}"
)
display(edges[EDGE_COLS].head(8))
for key in [
    ("cart", "cart"),
    ("checkout_start", "home"),
    ("checkout_start", "payment"),
]:
    print(" -> ".join(key), ":", edge_rows.loc[key, "reading"])
untested = edges[edges["verdict"] == "insufficient_data"]
print("not tested:", list(zip(untested["source"], untested["target"])))

90 of the 92 edges are tested and 19 differ after Benjamini-Hochberg. With the default
`edge_weight="unique_paths"` a row is the share of the arm's assigned users whose path has the
transition (`group1_edge_paths` of `group1_paths`), tested with the scorecard's pooled z-test.
Most discoveries restate the feature: `quick_add` (treatment only, `one_arm_only`) replaces
`add_to_cart`. Two are about behaviour: the `cart -> cart` self-loop exists in treatment only
(+4.78 pp, the planted cart loop), and `checkout_start -> home` triples (+1.30 pp), while
`checkout_start -> payment` does not move (+0.06 pp, [-1.71, +1.84]). Each discovery's `reading`
ends by calling it a screen result, and an inconclusive row's names the family (`screen: bh over
90 tested edges`); its `mde` is computed at `alpha / n_tested`. The two edges below the pooled
support of `min_support=10` paths are reported, not tested, and are outside the family.

`transition_graph(diff=..., ab_transitions=frame)` draws the frame on the diff graph:
significant edges in their sign's colour (red where treatment is higher), inconclusive edges
grey, untested ones dashed, edges without a row in the frame faint; each edge of the frame is
labelled with its delta in pp, and the edge tooltip shows the
test's values (here the arms' shares, not the graph's path counts), the delta's interval, the
adjusted p-value and the verdict, and a significant edge stays visible whatever the edge
filter. The legend names the correction, the 90 tested edges and ITT. The frame must have the
graph's `diff`, `edge_weight` and `path_col`. The graph draws the stream's paths and
the overlay is the ITT test, whose shares also count the 267 silent users in their arms'
denominators.

In [ ]:
stream.transition_graph(diff=DIFF, ab_transitions=edges).render_static()

**`edge_weight=`.** Five weights are tested, each with the scorecard's test for its kind:
`unique_paths` (0/1 per path, z-test), `avg_per_path` (the transition's count per path, Welch)
and three ratios of sums tested with the joint path bootstrap: `proba_out` (over the
transitions out of `source`), `proba_in` (into `target`) and `share_of_total`. The ratios'
denominators are behaviour the treatment changes, so every ratio row is `conditional=True`.
`count` is refused (it grows with the arm's size), and so are the time weights.

In [ ]:
by_weight = []
for weight in [
    "unique_paths",
    "avg_per_path",
    "proba_out",
    "proba_in",
    "share_of_total",
]:
    # n_bootstrap applies to the bootstrapped (ratio) weights only
    boot = (
        {"n_bootstrap": N_BOOT}
        if weight in ("proba_out", "proba_in", "share_of_total")
        else {}
    )
    frame = stream.get_ab_transitions(DIFF, weight, assignment=assignment, **boot)
    row = frame.set_index(["source", "target"]).loc[("checkout_start", "payment")]
    by_weight.append(
        {
            "edge_weight": weight,
            "n_tested": frame.attrs["n_tested"],
            "discoveries": int(frame["verdict"].isin(["increase", "decrease"]).sum()),
            "test": row["test"],
            "conditional": row["conditional"],
            "group1_value": row["group1_value"],
            "group2_value": row["group2_value"],
            "delta": row["delta"],
            "delta_ci_low": row["delta_ci_low"],
            "delta_ci_high": row["delta_ci_high"],
            "verdict": row["verdict"],
        }
    )
    if weight == "proba_out":
        proba_out_edges = frame
weights = pd.DataFrame(by_weight).set_index("edge_weight")
print("checkout_start -> payment under every weight:")
display(weights)
print(
    "proba_out reading:",
    proba_out_edges.set_index(["source", "target"]).loc[
        ("checkout_start", "payment"), "reading"
    ],
)
show_error(stream.get_ab_transitions, DIFF, "count")
show_error(stream.get_ab_transitions, DIFF, "time_median")

The same edge, two answers. Per assigned user, the share with a checkout followed by a payment
is unchanged (`unique_paths`, `avg_per_path`). Out of the checkouts, the share followed by a
payment falls by 5.25 pp (`proba_out`, `decrease`): the treatment's "checkout now" shortcut adds
checkouts that nobody meant to finish, so the denominator grew while the payments did not. That
is the conditional funnel step of section 6, now with a p-value, and the row says what it is:
`conditional=True`, and a reading that names the denominator and says it compares transitions,
not the arms. Read the per-user weights for an effect of the treatment.

**`edges=`, `correction=`, `horizon=`, and the graph's numbers.** Edges declared before the test
form a confirmatory family of their own (the rows for them only); `correction` and `ci_level`
work as in the scorecard; `horizon` keeps each user's first days after assignment and leaves out
the users whose window is open, the same users as the scorecard's. On the stream's paths, an
arm's value is the number `transition_graph_data` draws for it.

In [ ]:
declared = stream.get_ab_transitions(
    DIFF,
    assignment=assignment,
    edges=[("cart", "cart"), ("checkout_start", "payment")],
    correction="holm",
)
display(
    declared[
        [
            "source",
            "target",
            "delta",
            "p_value",
            "p_value_adjusted",
            "correction_family",
            "verdict",
        ]
    ]
)
edges_28 = stream.get_ab_transitions(DIFF, assignment=assignment, horizon="28D")
print(
    f"horizon 28D: {edges_28.attrs['n_tested']} edges tested, immature paths "
    f"{edges_28['immature_paths'].iloc[0]} (scorecard: {card_28['immature_paths'].iloc[0]})"
)
on_stream = stream.get_ab_transitions(DIFF, "proba_out", n_bootstrap=N_BOOT)
_, graph_treatment, graph_control = stream.transition_graph_data("proba_out", diff=DIFF)
stream_row = on_stream.set_index(["source", "target"]).loc[
    ("checkout_start", "payment")
]
graph_values = (
    graph_treatment.loc["checkout_start", "payment"],
    graph_control.loc["checkout_start", "payment"],
)
print(
    "proba_out checkout_start -> payment on the stream:",
    (round(stream_row["group1_value"], 6), round(stream_row["group2_value"], 6)),
    "| transition_graph_data:",
    tuple(round(float(v), 6) for v in graph_values),
)

**`get_ab_steps(diff, max_steps=...)`.** One row per cell `(event, step)`: the share of the
arm's paths whose k-th event is the event (`path_end` for a path that ended before step k; a
silent user has ended at step 1), the scorecard's z-test, Benjamini-Hochberg over the tested
cells. Steps count from each path's start; to compare around an event, cut the paths with
`truncate_paths` first (there is no `path_pattern`).

In [ ]:
cells = stream.get_ab_steps(DIFF, max_steps=8, assignment=assignment)
print(
    f"{cells.attrs['n_tested']} cells tested | {cells['verdict'].value_counts().to_dict()} | "
    f"family {cells['correction_family'].iloc[0]}"
)
display(
    cells[cells["verdict"].isin(["increase", "decrease"])][
        [
            "event",
            "step",
            "group1_value",
            "group2_value",
            "delta",
            "delta_ci_low",
            "delta_ci_high",
            "p_value_adjusted",
            "verdict",
        ]
    ]
)
show_error(stream.get_ab_steps, DIFF, path_pattern="cart")

# The A/A experiment: a true null for every edge and every cell
aa_edges = aa.get_ab_transitions(DIFF, assignment=aa_assignment)
aa_cells = aa.get_ab_steps(DIFF, assignment=aa_assignment)
print(
    f"A/A: {aa_edges.attrs['n_tested']} edges tested, "
    f"{int(aa_edges['verdict'].isin(['increase', 'decrease']).sum())} discoveries; "
    f"{aa_cells.attrs['n_tested']} cells tested, "
    f"{int(aa_cells['verdict'].isin(['increase', 'decrease']).sum())} discoveries"
)

93 cells are tested and 14 differ: `quick_add` is higher and `add_to_cart` lower at every step
from 3 to 8, `cart` at step 8 is higher by +1.71 pp (the loop: a user who reopens the cart is
still in it later), and `search` at step 6 is lower. Cells are not independent (a path that
reaches the cart one step earlier moves its `cart` cells at two steps, and a path one event
longer moves a `path_end` cell), so read the screen as a map of where the paths part; and the
cells that pass are selected for a large delta, so their deltas sit further from 0 than the
true effects (the winner's curse). On the A/A experiment, where both
arms run the same build, neither screen finds anything. One A/A run is a smoke test; over 20 A/A
seeds the edge screen made no discovery in any run, and the step screen one cell in one run.

**The screens as matrices: `ab_transitions` and `ab_steps`.** These widgets draw the
same frames as a source × target and an event × step matrix, through their twins
`ab_transitions_data` and `ab_steps_data` (documented aliases of `get_ab_transitions` and
`get_ab_steps`). Every cell is one row of the frame: the background is the delta (red where
treatment is higher, blue where it is lower, the diff graph's colours), the text the delta and
its per-cell interval in pp, and a wide outline reads `p_value_adjusted`: black below alpha,
light grey with a grey cross when tested and not significant, dashed when not tested (the two
edges under `min_support`). An empty cell has no row: the transition never occurs. Hover a cell
for the whole row and its reading. In the live widget the sidebar recomputes `edge_weight` (or
`max_steps`), `correction`, `ci_level` and `min_support`; the view controls (hide the rows
without a significant cell, clip the colour scale, sort, pin, hide an event) do not recompute.

In [ ]:
edge_matrix = stream.ab_transitions(DIFF, assignment=assignment)
edge_matrix.render_static()

In [ ]:
cell_matrix = stream.ab_steps(DIFF, max_steps=8, assignment=assignment)
cell_matrix.render_static()

In [ ]:
def outline(row, alpha):
    # the matrices' outline: black below alpha, grey when tested, dashed when not
    p = row["p_value_adjusted"]
    if p is None:
        return "dashed"
    return "black" if p < alpha else "grey"


edge_payload = json.loads(edge_matrix.result)
cell_payload = json.loads(cell_matrix.result)
matrix_twins_equal = stream.ab_transitions_data(DIFF, assignment=assignment).equals(
    edges
) and stream.ab_steps_data(DIFF, max_steps=8, assignment=assignment).equals(cells)
outlines = {
    name: pd.Series([outline(row, payload["alpha"]) for row in payload["rows"]])
    .value_counts()
    .to_dict()
    for name, payload in [("edges", edge_payload), ("cells", cell_payload)]
}
print(
    "ab_transitions_data == get_ab_transitions and ab_steps_data == get_ab_steps:",
    matrix_twins_equal,
)
print(
    f"{len(edge_payload['rows'])} edges and {len(cell_payload['rows'])} cells drawn | "
    f"outlines {outlines}"
)

**As tables for export: `get_ab_transitions_table` and `get_ab_steps_table`.** The same
frames, pivoted into the shape of the diff matrices: a row per source (per event) and, per
target (per step), a group of statistic columns, by default `delta`, `delta_ci_low`,
`delta_ci_high` and `p_value_adjusted` (`values=` takes any statistic column of the frame).
Nothing is recomputed, a cell without a row is NaN, and the frame's `attrs` come along.
`to_csv` writes the two header rows and `read_csv(header=[0, 1], index_col=0)` reads them
back; `styled=True` returns a pandas `Styler` with the matrices' colours and outlines, which
`to_excel` keeps (with `openpyxl` installed).

In [ ]:
edge_table = stream.get_ab_transitions_table(DIFF, assignment=assignment)
print(edge_table.shape, "| values:", edge_table.attrs["values"])
display(
    edge_table.loc[
        ["add_to_cart", "cart", "checkout_start"], ["cart", "home", "payment"]
    ]
)
table_path = Path(tempfile.mkdtemp()) / "edges.csv"
edge_table.to_csv(table_path)
read_back = pd.read_csv(table_path, header=[0, 1], index_col=0)
csv_round_trip = np.allclose(
    read_back.to_numpy(dtype=float), edge_table.to_numpy(dtype=float), equal_nan=True
)
print("CSV round trip:", csv_round_trip, "| header rows:", read_back.columns.nlevels)
stream.get_ab_steps_table(
    DIFF,
    max_steps=8,
    assignment=assignment,
    values=["delta", "p_value_adjusted"],
    styled=True,
)

## 8. The `ab_test` widget

`stream.ab_test(...)` takes the arguments of `get_ab_test` and draws its rows: a Scorecard tab
(a table with each arm's value, the delta and its interval, the adjusted p-value, the MDE, the
verdict, any `support_warning` and the `reading`, next to a forest plot of the deltas coloured
by verdict), a Strata tab with `strata_col`, and a Periods tab with `period`. Its headless twin
is `ab_test_data`, a documented alias of `get_ab_test`, so the widget's numbers are the getter's.
In the live widget the sidebar edits the arms, the metrics, `ci_level`, `correction`, `test`,
`strata_col`, `period`, `period_basis` and `horizon`, and recomputes. Here it is rendered once,
with the default sidebar (open):

In [ ]:
widget = stream.ab_test(
    DIFF,
    [CONVERSION, REVENUE, AOV],
    assignment=assignment,
    strata_col="platform",
    strata_pre_exposure=True,
    n_bootstrap=N_BOOT,
)
widget.render_static()

In [ ]:
headless = stream.ab_test_data(
    DIFF, [CONVERSION, REVENUE, AOV], assignment=assignment, n_bootstrap=N_BOOT
)
getter = stream.get_ab_test(
    DIFF, [CONVERSION, REVENUE, AOV], assignment=assignment, n_bootstrap=N_BOOT
)
pd.testing.assert_frame_equal(headless, getter)
drawn = pd.DataFrame(json.loads(widget.result)["overall"])
same_numbers = np.allclose(
    drawn[["delta", "delta_ci_low", "delta_ci_high"]].to_numpy(dtype=float),
    getter[["delta", "delta_ci_low", "delta_ci_high"]].to_numpy(),
)
print(
    "ab_test_data == get_ab_test:",
    headless.equals(getter),
    "| widget rows == get_ab_test:",
    same_numbers,
)
print(
    "widget payload:", sorted(json.loads(widget.result)), "| error:", repr(widget.error)
)

**`state_file=`, `height=`, `sidebar_open=`.** `height` (default 560 px) and `sidebar_open`
(default True) set the frame. `state_file=` binds the widget's state (its data parameters and
both display parameters) to a JSON file: created on first use, auto-saved on every sidebar
change, and loaded back by any later widget built with the same file, so `diff` and `metrics`
may be left out. The assignment table is never saved: pass it again.

In [ ]:
state_path = Path(tempfile.mkdtemp()) / "quick_add_scorecard.json"
saved = stream.ab_test(
    DIFF,
    [CONVERSION],
    assignment=assignment,
    horizon="28D",
    state_file=str(state_path),
    height=480,
    sidebar_open=False,
)
restored = stream.ab_test(
    assignment=assignment, state_file=str(state_path)
)  # no diff, no metrics
print(
    "state saved:",
    state_path.exists(),
    "| keys:",
    sorted(json.loads(state_path.read_text())["state"]),
)
print(
    "restored: diff",
    restored.diff,
    "| horizon",
    restored.horizon,
    "| height",
    restored.height,
    "| sidebar_open",
    restored.sidebar_open,
    "| same rows:",
    restored.result == saved.result,
)

## 9. Asking an agent: the MCP tools

The library's MCP server lets an AI assistant connected through any MCP client ask the same
questions. From a notebook:

```python
from retentioneering.mcp import serve
from retentioneering import datasets

stream = datasets.load_ab_test()
serve(stream, assignment=datasets.load_ab_test_assignment())   # SSE on port 8765
```

and in the client's MCP configuration:

```json
{"mcpServers": {"retentioneering": {"url": "http://localhost:8765/sse"}}}
```

With `assignment=`, the A/B tools count every assigned user (ITT) and `horizon=` /
since-assignment periods become available; the table stays in the notebook's process. The six
A/B tools are `get_sample_ratio`, `get_ab_test` and `get_ab_bootstrap` (numbers only, a
contrast summary rather than the replicate table), `get_ab_transitions` and `get_ab_steps` (the
screens of section 7: counts per verdict and the top rows), and `add_ab_test` (registers the
scorecard as a report tab); `get_ab_paths` is Python only. The playbook's `ab_test` scenario tells the agent
the order: sample ratio first, a decision metric declared before the first scorecard, an
inconclusive row read with its interval and `mde`.

The cell below builds the server **without starting it** and lists what is registered, then
calls the tool functions behind it directly, so you can see what an agent receives.
(`_build_server` and `ReportSession` are internals, what `serve()` runs; they are used here only
to look inside without opening a port.)

In [ ]:
from retentioneering.mcp import server as mcp_server
from retentioneering.mcp import tools as mcp_tools
from retentioneering.mcp._report_session import ReportSession

# Build the server serve() would run, without starting it: nothing listens on a port.
built = mcp_server._build_server(stream, {}, assignment=assignment)
registered = [tool.name for tool in built._tool_manager.list_tools()]
AB_TOOLS = [
    "get_sample_ratio",
    "get_ab_test",
    "get_ab_bootstrap",
    "get_ab_transitions",
    "get_ab_steps",
    "add_ab_test",
]
print("registered tools:", registered)
print(
    "A/B tools present:",
    all(name in registered for name in AB_TOOLS),
    "| registered prompts:",
    len(built._prompt_manager.list_prompts()),
)
print(
    "system instructions mention the table:",
    [
        line
        for line in built.instructions.splitlines()
        if "assignment table is attached" in line
    ],
)
print(
    "describe()['assignment'] ->",
    mcp_tools.describe(ReportSession(stream, {}, assignment=assignment), {})[
        "assignment"
    ],
)
print(mcp_tools.playbook("ab_test")[:400], "...")

In [ ]:
session = ReportSession(
    stream, {}, assignment=assignment
)  # what serve(stream, assignment=...) holds
agent_srm = mcp_tools.get_sample_ratio(session, "exp_group")
agent_card = mcp_tools.get_ab_test(session, list(DIFF), [CONVERSION])
agent_novelty = mcp_tools.get_ab_bootstrap(
    session,
    list(DIFF),
    [CONVERSION],
    period="14D",
    period_basis="since_assignment",
    horizon="42D",
    contrast={"novelty": [["0-13"], ["28-41"]]},
)
agent_edges = mcp_tools.get_ab_transitions(session, list(DIFF), top=3)
agent_tab = mcp_tools.add_ab_test(session, "Scorecard", list(DIFF), [CONVERSION])
print("get_sample_ratio ->", agent_srm["reading"])
print(
    "get_ab_test ->",
    json.dumps(
        {
            k: agent_card["metrics"][0][k]
            for k in ["metric", "population", "delta", "verdict", "reading"]
        },
        ensure_ascii=False,
    ),
)
print("  scope:", agent_card["metrics"][0]["scope"])
print("get_ab_bootstrap ->", json.dumps(agent_novelty["rows"][0]))
print(
    "get_ab_transitions ->",
    {k: agent_edges[k] for k in ["n_tested", "n_increase", "n_decrease", "population"]},
    [(row["source"], row["target"], row["verdict"]) for row in agent_edges["top"]],
)
print(
    "add_ab_test -> tab",
    agent_tab["tab_id"],
    "| pending tabs:",
    len(session.pending_tabs),
)

The agent receives the same numbers as the notebook: the sample ratio `reading`, the scorecard
row with its `reading` and a `scope` sentence saying what was corrected, and the novelty
contrast with its point estimate (+4.86 pp, the difference of the two `horizon="42D"` rows),
the interval from 1,000 replicates and `share_positive`, and the edge screen's counts with its
top rows. `export_report` saves only report
tabs, which is why `add_ab_test` exists.

## 10. Blind pre-exposure clusters as strata

The short version of the tutorial's step 6, on the default dataset. A grey top line can hide
users who gain and users who lose. To look for them without fooling ourselves, the groups must
be fixed **before exposure**: here, the part of each path before the first `product_view` (the
button sits on the product page). Cluster users on that window blind to the arm and the
outcome, name the clusters after their marker events, carry the labels back with the
declaration, and stratify.

In [ ]:
window_balance = pre_exposure.get_ab_test(
    DIFF, [{"name": "window_length", "metric": "length"}], assignment=assignment
)
print("the window before the first product_view, by arm (it must not differ):")
display(window_balance[CORE])

clustered = pre_exposure.add_clusters(
    "behaviour",
    features=[{"metric": "has_event_bulk"}, {"metric": "active_days"}],
    method="kmeans",
    method_args={"n_clusters": 3, "n_init": 4, "random_state": 42},
    scaler="minmax",
)
labels = clustered.get_path_segments("behaviour")
flags = pre_exposure.get_metrics([{"metric": "has_event_bulk"}])
flags.columns = flags.columns.str.removeprefix("has_event_bulk_")
marker = (flags.groupby(labels).mean() / flags.mean()).idxmax(axis=1)
cluster_names = {
    label: {"search_filter": "scanner", "collections_view": "curator"}.get(
        event, "other"
    )
    for label, event in marker.items()
}
print("markers:", marker.to_dict(), "->", cluster_names)

with_clusters = stream.add_segment(
    "behaviour", path_values=labels.map(cluster_names), pre_exposure=True
)
cluster_strata = with_clusters.get_ab_test(
    DIFF, [CONVERSION], assignment=assignment, strata_col="behaviour"
)
first_cluster = cluster_strata.iloc[0]
print(
    f"balance p {first_cluster['balance_p_value']:.3f}, heterogeneity p {first_cluster['heterogeneity_p_value']:.2g}"
)
cluster_strata[strata_cols]

Scanners (search with filters before their first product page) convert about 10 pp more with
`quick_add`, curators (collections and wishlists) about 5 pp less, both after Holm, and
Cochran's Q says the strata differ. The rows read `strata_pre_exposure_source="declared"`
because `add_segment(..., pre_exposure=True)` recorded our declaration. Which users gain is a
hypothesis for a targeted follow-up test, not a launch: the window, K and the features were
choices, and the largest of several stratum effects is biased upwards.

Against the planted truth (`with_truth=True`, used here and nowhere else): the same scorecard
stratified by the planted user types, next to the blind clusters.

In [ ]:
by_truth = stream.add_segment(
    "planted_type", path_values=planted_types, pre_exposure=True
).get_ab_test(DIFF, [CONVERSION], assignment=assignment, strata_col="planted_type")
truth_vs_clusters = (
    by_truth.set_index("stratum")[["delta", "delta_ci_low", "delta_ci_high", "verdict"]]
    .add_prefix("planted_")
    .join(
        cluster_strata.set_index("stratum")[
            ["delta", "delta_ci_low", "delta_ci_high", "verdict"]
        ].add_prefix("cluster_")
    )
)
print("crosstab, clusters (rows) vs planted types (columns):")
display(pd.crosstab(labels.map(cluster_names), planted_types.reindex(labels.index)))
truth_vs_clusters

The planted-type rows reproduce the effects the generator documents for this seed (+12.5 pp
for scanners, -6.7 pp for curators, -0.4 pp for the others). Every planted scanner and curator
lands in the cluster of that name, and the signs and verdicts agree; the cluster deltas are
smaller (+10.0 and -5.1 pp) because the scanner and curator clusters also take in some planted
`other` users, which pulls their deltas toward zero.

## 11. Checks

The cell below asserts every claim the prose of this notebook makes about the printed results,
so the text cannot drift away from the numbers.

In [ ]:
def verdicts(frame, key="metric"):
    return dict(zip(frame[key], frame["verdict"]))


def r2(value):
    return round(float(value) * 100, 2)


# 1. datasets
assert list(catalogue) == [
    "northwind_home",
    "srm_platform_bug",
    "session_assignment",
    "aa_calibration",
    "trial_conversion",
    "push_campaign",
]
assert len(assignment) == 10_000 and stream.df["user_id"].nunique() == 9_733
assert draw.equals(same) and not draw.equals(other) and len(draw_assignment) == 500
assert session_table.index.name == "session_id" and assignment.index.name == "user_id"

# 2. sample ratio
assert (
    not srm["mismatch"]
    and srm["silent_paths"] == 267
    and srm["multi_valued_paths"] == 0
)
assert round(srm["p_value"], 2) == 0.42 and wrong_design["mismatch"]
assert (
    not by_platform["strata"]["mismatch"].any()
    and not by_week["periods"]["mismatch"].any()
)
assert bug_srm["mismatch"] and set(bug_srm["strata"].query("mismatch")["stratum"]) == {
    "android"
}
assert bug_weeks["periods"]["mismatch"].tolist() == [False, True, True, True]
assert len(bug_daily["periods"]) == 28 and not bug_daily["periods"]["mismatch"].any()
assert round(treatment_share[treatment_share.index < "2026-05-12"].mean(), 2) == 0.50
assert round(treatment_share[treatment_share.index >= "2026-05-12"].mean(), 2) == 0.42
assert (
    verdicts(naive)["conversion"] == "increase"
    and round(naive["delta"].iloc[0] * 100) == 3
)
assert verdicts(by_bug_platform, "stratum") == {
    "android": "increase",
    "ios": "inconclusive",
    "web": "inconclusive",
}
assert (
    per_user["mismatch"]
    and per_user["multi_valued_paths"] == 2454
    and round(per_user["p_value"], 2) == 0.83
)
assert not per_session["mismatch"]
assert push_stream_srm["mismatch"] and not push_srm["mismatch"]
assert push_stream_srm["levels"].set_index("level")["paths"].to_dict() == {
    "control": 2557,
    "treatment": 3680,
}

# 3. scorecard
assert set(scorecard["population"]) == {"assignment"}
assert (scorecard["group1_paths"] == 4960).all() and (
    scorecard["group2_paths"] == 5040
).all()
_conditional = set(scorecard.loc[scorecard["conditional"], "metric"])
assert _conditional == {
    "path_duration_s",
    "time_to_order_s",
    "mean_order_value",
    "last_order_value",
    "aov",
}
assert scorecard.set_index("metric").loc[
    "time_to_order_s", ["group1_support", "group2_support"]
].tolist() == [1207, 1196]
assert (
    scorecard.set_index("metric").loc["cart_loop", "support_warning"] == "one_arm_only"
)
assert verdicts(scorecard)["mobile_user"] == "inconclusive"
assert (
    r2(conversion_row["delta_ci_low"]),
    r2(conversion_row["delta_ci_high"]),
    r2(conversion_row["mde"]),
) == (-1.07, 2.28, 2.39)
assert (
    "5.7× the paths" in conversion_row["reading"]
    and round(conversion_row["rel_delta"], 3) == 0.025
)
assert verdicts(per_metric)["revenue_per_user"] == "inconclusive"
assert per_metric.set_index("metric").loc["revenue_per_user", "test"] == "bootstrap"
assert comparison["verdict_holm"].tolist()[1:3] == ["increase", "inconclusive"]
assert comparison["verdict_bh"].tolist()[1:3] == ["increase", "increase"]
assert comparison["verdict_none"].tolist()[1:3] == ["increase", "increase"]
assert comparison["delta_ci_low"].iloc[2] > 0
assert (
    round(comparison["p_holm"].iloc[2], 3),
    round(comparison["p_bonferroni"].iloc[2], 3),
) == (0.057, 0.065)
assert (
    r2(bonferroni["delta_ci_low_adj"].iloc[2]),
    r2(bonferroni["delta_ci_high_adj"].iloc[2]),
) == (-0.05, 2.61)
assert set(scopes.query("scope == 'all'")["correction_family"]) == {"all_metrics"}
assert guardrail_rows["passes"].tolist() == [True, False]
assert round(guardrail_rows.loc["events_per_user", "delta_ci_low"], 2) == -1.07
assert [round(x * 100, 2) for x in pd.DataFrame(levels)["mde"]][::3] == [2.39, 2.77]
assert verdicts(renamed) == {
    "add_to_cart_users": "decrease",
    "quick_add_users": "increase",
    "adders": "inconclusive",
}
assert (
    renamed.set_index("metric").loc["quick_add_users", "support_warning"]
    == "one_arm_only"
)
assert round(renamed.set_index("metric").loc["quick_add_users", "delta"] * 100) == 51
assert (
    platform_strata.iloc[0]["support_warning"] == "no_data"
    and platform_strata.iloc[0]["stratum"] == "<MISSING>"
)
assert low.set_index("metric").loc["aov", "support_warning"] == "low_support"
assert low.set_index("metric").loc[
    "aov", ["group1_support", "group2_support"]
].tolist() == [31, 19]
assert set(tiny_default["verdict"]) == {"insufficient_data"}
assert "pooled support 7 < min_support 10" in tiny_default["reading"].iloc[0]
assert verdicts(tiny_min1) == {"conversion": "inconclusive", "aov": "inconclusive"}
assert (
    tiny_min1.set_index("metric").loc["aov", "support_warning"] == "invalid_replicates"
)
assert tiny_min1.set_index("metric").loc[
    "aov", ["group1_support", "group2_support"]
].tolist() == [4, 3]
assert np.isnan(tiny_min1.set_index("metric").loc["aov", "p_value"])
# strata provenance
assert (
    platform_strata["strata_pre_exposure_source"] == "declared"
).all() and not platform_strata["exploratory"].any()
assert (
    undeclared["strata_pre_exposure_source"] == "not_pre_exposure"
).all() and undeclared["exploratory"].all()
assert round(platform_strata["heterogeneity_p_value"].iloc[0], 2) == 0.94
assert set(platform_strata["correction_family"]) == {"metric=conversion/strata"}
assert window_labels.attrs.get("pre_exposure") is True
assert (
    recipe_strata["strata_pre_exposure_source"] == "recipe"
).all() and not recipe_strata["exploratory"].any()
assert (
    post_strata["strata_pre_exposure_source"] == "not_pre_exposure"
).all() and post_strata["exploratory"].all()
assert round(post_strata["balance_p_value"].iloc[0], 2) == 0.33
assert (
    "<MISSING>" not in set(push_strata["stratum"])
    and push_strata["group1_silent_paths"].sum() > 0
)
# diff forms
_forms = pd.DataFrame(form_rows).set_index("form")
assert (
    _forms.loc["segment + table", "population"] == "assignment"
    and _forms.loc["segment, no table", "population"] == "stream"
)
assert _forms.loc["segment, no table", ["group1_paths", "group2_paths"]].sum() == 9733
assert _forms.loc["segment + table", "silent"] == 132
for _form in ["'arm' + table", "Series table", "table with string ids"]:
    assert _forms.loc[_form, "delta"] == _forms.loc["segment + table", "delta"]
assert _forms.loc["id lists", "delta"] == _forms.loc["segment, no table", "delta"]
# unit guard and session_assignment
assert verdicts(per_session_card)["add_rate"] == "increase"
assert (
    1.5 <= clustered_se / naive_se <= 1.8 and round(clustered_se / naive_se, 2) == 1.55
)
assert (
    round(first_session_card["delta"].iloc[0] * 100, 1) == 7.3
    and first_session_card["population"].iloc[0] == "stream"
)

# 4. time
assert (
    weekly["delta"].iloc[0] > 0 > weekly["delta"].iloc[-1]
    and not weekly["period_complete"].iloc[-1]
)
assert (
    set(weekly_active["verdict"]) == {"descriptive"}
    and weekly_active["p_value_adjusted"].isna().all()
)
assert [round(x * 100, 2) for x in horizons["delta"]] == [0.6, 2.75, 2.09, 1.0]
assert horizons["verdict"].tolist() == [
    "inconclusive",
    "increase",
    "increase",
    "inconclusive",
]
assert verdicts(since, "period") == {
    "0-13": "increase",
    "14-27": "inconclusive",
    "28-41": "decrease",
}
_late = since.set_index("period").loc["28-41"]
assert (
    round(_late["delta"] * 100, 2) == -1.48
    and round(_late["p_value_adjusted"], 3) == 0.023
)
assert (since["group1_paths"] + since["group2_paths"]).tolist() == [9995, 8702, 6336]
assert (
    since_42["group1_paths"] + since_42["group2_paths"]
).nunique() == 1 and since_42["period_complete"].all()
assert (
    round(novelty_point * 100, 2) == 4.86
    and 0 < novelty_low < novelty_point < novelty_high
)
_boot = pd.DataFrame(boot_rows).set_index("call")
assert (
    _boot.loc["multinomial, seed 7", "delta_ci_low"]
    == _boot.loc["multinomial, seed 7 again", "delta_ci_low"]
)
assert (
    _boot.loc["multinomial, seed 7", "delta_ci_low"]
    != _boot.loc["multinomial, seed 42", "delta_ci_low"]
)
assert _boot.loc["n_bootstrap='auto'", "valid_replicates"] == 1000 and auto_strata[
    "valid_replicates"
].tolist() == [0, 1200, 1200]
_mmd = mobile_minus_desktop.quantile([0.025, 0.975])
assert _mmd.iloc[0] < 0 < _mmd.iloc[1]
assert np.allclose(
    aov_reps["delta"].quantile([0.025, 0.975]),
    [aov_card["delta_ci_low"], aov_card["delta_ci_high"]],
)
# trial
assert (
    verdicts(trial_open)["subscribed"] == "increase"
    and round(trial_open["delta"].iloc[0] * 100, 1) == 3.3
)
assert (
    verdicts(trial_21)["subscribed"] == "inconclusive"
    and trial_21["immature_paths"].iloc[0] == 4410
)
assert round(trial_21["delta"].iloc[0] * 100, 1) == 0.9
assert verdicts(trial_bins, "period") == {
    "0-6": "increase",
    "7-13": "decrease",
    "14-20": "decrease",
}
assert shift.iloc[0] > 0
# aa
assert verdicts(aa_card) == {
    "conversion": "inconclusive",
    "orders_per_user": "inconclusive",
    "revenue_per_user": "increase",
    "aov": "increase",
    "revenue_bootstrap": "increase",
}
assert set(aa_daily["verdict"]) == {"inconclusive"}
assert set(top_revenue.index[:6]) == set(whales) and set(
    top_revenue["arm"].iloc[:6]
) == {"treatment"}
# push
assert verdicts(push_itt) == {"purchase": "increase", "opened_app": "increase"}
assert (
    round(push_itt["delta"].iloc[0] * 100, 2) == 1.94
    and round(push_naive["delta"].iloc[0] * 100, 2) == -2.27
)
assert verdicts(push_naive) == {
    "purchase": "decrease",
    "opened_app": "insufficient_data",
}

# 5. per-path table
assert np.allclose(joined["group1_value"], joined["treatment_from_paths"])
assert np.allclose(joined["group2_value"], joined["control_from_paths"])
assert int((~paths["mature"]).sum()) == card_28["immature_paths"].iloc[0]

# 6. funnels
assert (funnel_table["delta_conversion_rate_p_value_adjusted"] > 0.05).all()
assert funnel["correction_family"] == "metric=delta_conversion_rate/steps"
assert funnel_table.loc["checkout_start", "delta_step_conversion_rate_ci_low"] > 0
assert funnel_table.loc["payment", "delta_step_conversion_rate_ci_high"] < 0
assert round(funnel_table.loc["order_success", "rel_delta_conversion_rate"], 3) == 0.025
assert (
    round(funnel_table.loc["order_success", "delta_conversion_rate"] * 100, 2) == 0.62
)
assert pairs["conditional"].tolist() == [False, True]

# 7. edges and steps
assert edges.attrs["n_tested"] == 90 and len(edges) == 92
assert edges["verdict"].value_counts().to_dict() == {
    "inconclusive": 71,
    "increase": 10,
    "decrease": 9,
    "insufficient_data": 2,
}
assert set(edges["population"]) == {"assignment"}
assert edges["correction_family"].iloc[0] == "edges=all/unique_paths"
_loop_edge = edge_rows.loc[("cart", "cart")]
assert (
    _loop_edge["verdict"] == "increase"
    and _loop_edge["support_warning"] == "one_arm_only"
    and _loop_edge["group2_edge_paths"] == 0
    and r2(_loop_edge["delta"]) == 4.78
)
assert edge_rows.loc[("product_view", "quick_add"), "support_warning"] == "one_arm_only"
assert edge_rows.loc[("product_view", "add_to_cart"), "verdict"] == "decrease"
assert edge_rows.loc[("checkout_start", "home"), "verdict"] == "increase"
assert r2(edge_rows.loc[("checkout_start", "home"), "delta"]) == 1.30
assert (
    edge_rows.loc[("checkout_start", "home"), "group1_value"]
    > 3 * edge_rows.loc[("checkout_start", "home"), "group2_value"]
)
_pay = edge_rows.loc[("checkout_start", "payment")]
assert _pay["verdict"] == "inconclusive" and (
    r2(_pay["delta"]),
    r2(_pay["delta_ci_low"]),
    r2(_pay["delta_ci_high"]),
) == (0.06, -1.71, 1.84)
assert "screen result (bh over 90 tested edges)" in _loop_edge["reading"]
assert "screen: bh over 90 tested edges" in _pay["reading"]
assert len(untested) == 2 and set(untested["p_value_adjusted"].isna()) == {True}
assert weights.loc["unique_paths", "test"] == "z_test"
assert weights.loc["avg_per_path", "test"] == "welch_t_test"
assert set(weights.loc[["proba_out", "proba_in", "share_of_total"], "test"]) == {
    "bootstrap"
}
assert weights["conditional"].tolist() == [False, False, True, True, True]
assert verdicts(weights.reset_index(), "edge_weight")["unique_paths"] == "inconclusive"
assert verdicts(weights.reset_index(), "edge_weight")["avg_per_path"] == "inconclusive"
assert weights.loc["proba_out", "verdict"] == "decrease"
assert r2(weights.loc["proba_out", "delta"]) == -5.25
assert (
    "conditional on post-treatment behaviour"
    in proba_out_edges.set_index(["source", "target"]).loc[
        ("checkout_start", "payment"), "reading"
    ]
)
assert len(declared) == 2 and set(declared["correction_family"]) == {
    "edges=declared/unique_paths"
}
assert edges_28["immature_paths"].iloc[0] == card_28["immature_paths"].iloc[0]
assert np.allclose(
    [stream_row["group1_value"], stream_row["group2_value"]], graph_values, rtol=1e-12
)
assert cells.attrs["n_tested"] == 93
_cell_hits = cells[cells["verdict"].isin(["increase", "decrease"])]
assert len(_cell_hits) == 14
_by_cell = cells.set_index(["event", "step"])
for _step in range(3, 9):
    assert _by_cell.loc[("quick_add", _step), "verdict"] == "increase"
    assert _by_cell.loc[("add_to_cart", _step), "verdict"] == "decrease"
assert _by_cell.loc[("cart", 8), "verdict"] == "increase"
assert r2(_by_cell.loc[("cart", 8), "delta"]) == 1.71
assert _by_cell.loc[("search", 6), "verdict"] == "decrease"
assert not aa_edges["verdict"].isin(["increase", "decrease"]).any()
assert not aa_cells["verdict"].isin(["increase", "decrease"]).any()
# 7. the matrices and the tables
assert matrix_twins_equal
assert outlines["edges"] == {"grey": 71, "black": 19, "dashed": 2}
assert outlines["cells"] == {"grey": 79, "black": 14}
assert [row["delta"] for row in edge_payload["rows"]] == edges["delta"].tolist()
assert [row["delta"] for row in cell_payload["rows"]] == cells["delta"].tolist()
assert edge_table.shape == (16, 64) and edge_table.attrs["n_tested"] == 90
assert edge_table.loc["cart", ("cart", "delta")] == _loop_edge["delta"]
assert np.isnan(edge_table.loc["path_start", ("cart", "delta")])
assert csv_round_trip and read_back.columns.nlevels == 2

# 8. widget
assert widget.error == "" and same_numbers and headless.equals(getter)
assert (
    restored.horizon == "28D"
    and restored.height == 480
    and restored.sidebar_open is False
)

# 9. MCP
assert all(name in registered for name in AB_TOOLS)
assert agent_edges["n_tested"] == edges.attrs["n_tested"]
assert [(row["source"], row["target"]) for row in agent_edges["top"]] == list(
    zip(edges["source"].head(3), edges["target"].head(3))
)
assert agent_srm["reading"] == srm["reading"]
assert agent_card["metrics"][0]["delta"] == round(conversion_row["delta"], 6)
assert round(agent_novelty["rows"][0]["delta"] * 100, 2) == round(
    novelty_point * 100, 2
)

# 10. clusters
assert sorted(cluster_names.values()) == ["curator", "other", "scanner"]
assert verdicts(cluster_strata, "stratum") == {
    "<MISSING>": "insufficient_data",
    "curator": "decrease",
    "other": "inconclusive",
    "scanner": "increase",
}
assert round(cluster_strata.set_index("stratum").loc["scanner", "delta"] * 100) == 10
assert round(cluster_strata.set_index("stratum").loc["curator", "delta"] * 100) == -5
assert (
    cluster_strata.loc[
        cluster_strata["stratum"] != "<MISSING>", "strata_pre_exposure_source"
    ]
    == "declared"
).all()
assert cluster_strata["heterogeneity_p_value"].iloc[0] < 0.05
_cross = pd.crosstab(labels.map(cluster_names), planted_types.reindex(labels.index))
for _type in ("scanner", "curator"):
    assert _cross[_type].drop(index=_type).sum() == 0
assert [
    round(by_truth.set_index("stratum").loc[t, "delta"] * 100, 1)
    for t in ("scanner", "curator", "other")
] == [12.5, -6.7, -0.4]
assert truth_vs_clusters["planted_verdict"].equals(truth_vs_clusters["cluster_verdict"])
assert (
    np.sign(truth_vs_clusters["planted_delta"]).drop("<MISSING>").to_dict()
    == np.sign(truth_vs_clusters["cluster_delta"]).drop("<MISSING>").to_dict()
)
print("showcase checks passed")